# IMPORTS & CONFIG

In [ ]:
# =============================================================================
# CELL 1 — IMPORTS & CONFIG
# =============================================================================
# All dependencies and constants for the thesis analysis notebook.
# Requires: outputs/raw_df.csv, outputs/scored_monthly_base.csv,
#           outputs/fused_event_df.csv, debug_frames_qwen_anon_clean/
# =============================================================================

import os
import re
import ast
import json
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.image as mpimg
from pathlib import Path
from collections import Counter
from IPython.display import display

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS_DIR    = "outputs"
ANON_CLEAN_DIR = "debug_frames_qwen_anon_clean"

# ── Colour palette ────────────────────────────────────────────────────────────
C_BASE = "#2196F3"   # blue   = telematics baseline
C_VLM  = "#FF5722"   # orange = VLM adjusted
C_DIFF = "#4CAF50"   # green  = difference / improvement

# ── VLM detection config ──────────────────────────────────────────────────────
VLM_CODE_TO_EVENT_DESCRIPTION = {
    4:  "v_cam_covered",
    5:  "SEATBELT_D_OFF",
    6:  "v_cam_covered",
    10: "v_phone",
    11: "v_distraction",
    13: "v_fatigue",
    15: "v_smoke",
    66: "DRIVER_FACE_OBSTRUCTED",
    67: "v_cam_covered_review",   # tier 2 — possible covering
}

RANKABLE_VLM_EVENT_DESCRIPTIONS = {
    "v_cam_covered",
    "SEATBELT_D_OFF",
    "v_phone",
    "v_distraction",
    "v_fatigue",
    "v_smoke",
}

INCLUDE_REVIEW_EVENTS_IN_RANKING = False
RANKABLE_VLM_EVENTS_FOR_PIPELINE = sorted(RANKABLE_VLM_EVENT_DESCRIPTIONS)

print("✓ Config loaded")
#print(f"  Outputs dir    : {os.path.abspath(OUTPUTS_DIR)}")
#print(f"  Anon clean dir : {os.path.abspath(ANON_CLEAN_DIR)}")

# LOAD SAVED OUTPUTS

In [ ]:
# =============================================================================
# CELL 2 — LOAD SAVED OUTPUTS
# =============================================================================
# Loads raw_df, scored_monthly_base and fused_event_df saved from notebook 1.
# fused_event_df is fully built — no database or pipeline dependencies needed.
# =============================================================================

# ── Raw telematics data ───────────────────────────────────────────────────────
raw_df = pd.read_csv(os.path.join(OUTPUTS_DIR, "raw_df.csv"))
raw_df["event_ts"]   = pd.to_datetime(raw_df["event_ts"], utc=True, errors="coerce")
raw_df["vehicle_id"] = pd.to_numeric(raw_df["vehicle_id"], errors="coerce")
raw_df["terminal_event_id"] = (
    raw_df["terminal_event_id"].astype(str).str.split(".").str[0].str.strip()
)
raw_df["terminal_event_id"] = raw_df["terminal_event_id"].where(
    ~raw_df["terminal_event_id"].isin(["nan", "None", "NaT", ""]), other=None
)

# ── Telematics-only baseline rankings ─────────────────────────────────────────
scored_monthly = pd.read_csv(os.path.join(OUTPUTS_DIR, "scored_monthly_base.csv"))
scored_monthly["window_start"] = pd.to_datetime(scored_monthly["window_start"], errors="coerce")

# ── Fused event df — fully built in notebook 1 ───────────────────────────────
fused_event_df = pd.read_csv(
    os.path.join(OUTPUTS_DIR, "fused_event_df.csv"),
    dtype={"terminal_event_id": str}
)

# Restore list columns
for col in ["det_codes", "rev_codes", "det_event_descriptions", "rev_event_descriptions",
            "det_rankable_event_descriptions", "rev_rankable_event_descriptions",
            "extra_rankable_vlm_events"]:
    if col in fused_event_df.columns:
        fused_event_df[col] = fused_event_df[col].apply(
            lambda x: ast.literal_eval(x)
            if isinstance(x, str) and x.startswith("[") else
            (x if isinstance(x, list) else [])
        )

fused_event_df["event_ts_final"] = pd.to_datetime(
    fused_event_df.get("event_ts_final", pd.NaT), utc=True, errors="coerce"
)

print(f"raw_df                   : {raw_df.shape}")
print(f"scored_monthly           : {scored_monthly.shape}")
print(f"fused_event_df           : {fused_event_df.shape}")
print(f"Unique vehicles          : {raw_df['vehicle_id'].nunique()}")
print(f"Date range               : {raw_df['event_ts'].min()} → {raw_df['event_ts'].max()}")
print(f"\nChange type counts:")
display(
    fused_event_df["vlm_change_type"]
    .value_counts(dropna=False)
    .rename_axis("vlm_change_type")
    .reset_index(name="count")
)

# RANKING PIPELINE

In [ ]:
# =============================================================================
# CELL 3 — RANKING PIPELINE
# =============================================================================
# Copy of run_driver_ranking_pipeline from notebook 1.
# Required to rebuild scored_monthly_vlm from raw_df_vlm.
# =============================================================================

SESSION_GAP_SECONDS = (3 * 60 + 30)
NIGHT_START_HOUR = 22
NIGHT_END_HOUR = 5
HARSH_EPISODE_WINDOW_SECONDS = 5
SPEEDING_EPISODE_WINDOW_SECONDS = 180
POWER_EPISODE_WINDOW_SECONDS = 10 * 60
CAMERA_EPISODE_WINDOW_SECONDS = 60
FATIGUE_EPISODE_WINDOW_SECONDS = 30
DISTRACTION_EPISODE_WINDOW_SECONDS = 60
SITUATIONAL_EPISODE_WINDOW_SECONDS = 10
INSUFFICIENT_EXPOSURE_HOURS = 3.0

HARSH_SUBSTRINGS     = {"Corner": "harsh_corner_count", "Braking": "harsh_braking_count", "Accel": "harsh_accel_count"}
SPEEDING_SUBSTRINGS  = {"Speeding": "speeding_count"}
POWER_SUBSTRINGS     = {"Power OFF": "power_off_ext_batt_disc_count"}
CAMERA_SUBSTRINGS    = {"v_cam_covered": "cam_covered_count"}
FATIGUE_SUBSTRINGS   = {"v_eye_closed": "eye_closed_count", "v_yawn": "yawn_count", "v_fatigue": "fatigue_count"}
DISTRACTION_SUBSTRINGS = {"v_distraction": "distraction_count", "v_phone": "phone_use_count", "v_smoke": "smoke_count", "SEATBELT_D_OFF": "Driver_Seatbelt_not_on"}
SITUATIONAL_SUBSTRINGS = {"v_Headway_Mon": "headway_monitor_count", "v_lane_departure": "lane_departure_count", "v_fwd_collision": "forward_collision_count", "v_ped_collision": "ped_collision_count"}
CRASH_SUBSTRINGS     = {"CRASH": "crash_count"}

ALL_EVENT_COUNT_MAPS = (
    list(HARSH_SUBSTRINGS.items()) + list(SPEEDING_SUBSTRINGS.items())
    + list(POWER_SUBSTRINGS.items()) + list(CAMERA_SUBSTRINGS.items())
    + list(FATIGUE_SUBSTRINGS.items()) + list(DISTRACTION_SUBSTRINGS.items())
    + list(SITUATIONAL_SUBSTRINGS.items()) + list(CRASH_SUBSTRINGS.items())
)

agg_sum_cols = [
    "drive_seconds", "night_drive_seconds", "distance_km", "night_distance_km",
    "harsh_corner_count", "harsh_braking_count", "harsh_accel_count", "speeding_count",
    "power_off_ext_batt_disc_count", "cam_covered_count", "eye_closed_count", "yawn_count",
    "fatigue_count", "distraction_count", "phone_use_count", "smoke_count",
    "Driver_Seatbelt_not_on", "headway_monitor_count", "lane_departure_count",
    "forward_collision_count", "ped_collision_count", "crash_count",
    "harsh_episode_count", "short_speeding_episode_count", "long_speeding_episode_count",
    "speeding_episode_count", "power_violation_episode_count", "camera_obstruction_episode_count",
    "fatigue_episode_count", "driver_distraction_episode_count", "situational_risk_episode_count",
    "harsh_total_count", "fatigue_total_count", "driver_distraction_total_count", "situational_risk_total_count",
]

def _rank01(s):
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return s.rank(method="average", pct=True).fillna(0.0)

def _day_start(ts): return ts.floor("D")

def _split_interval_by_day(t0, t1):
    if t1 <= t0: return []
    parts, cur_start = [], t0
    while cur_start < t1:
        d0 = _day_start(cur_start)
        next_day = d0 + pd.Timedelta(days=1)
        cur_end = min(t1, next_day)
        parts.append((d0, cur_start, cur_end))
        cur_start = cur_end
    return parts

def _night_overlap_seconds(part_start, part_end):
    if part_end <= part_start: return 0.0
    d0 = _day_start(part_start)
    night1_start = d0 + pd.Timedelta(hours=0)
    night1_end   = d0 + pd.Timedelta(hours=NIGHT_END_HOUR)
    night2_start = d0 + pd.Timedelta(hours=NIGHT_START_HOUR)
    night2_end   = d0 + pd.Timedelta(days=1)
    def overlap(a0, a1, b0, b1):
        return max(0.0, (min(a1,b1) - max(a0,b0)).total_seconds())
    return overlap(part_start, part_end, night1_start, night1_end) + overlap(part_start, part_end, night2_start, night2_end)

def _to_month_start(d): return d.dt.to_period("M").dt.start_time

def _build_match_mask(desc_lower, substr_map):
    mask = pd.Series(False, index=desc_lower.index)
    for k in substr_map.keys():
        mask = mask | desc_lower.str.contains(k.lower(), na=False)
    return mask

def _add_raw_counts(df, desc_lower, counts_base, substr_map):
    out = counts_base.copy()
    for k, col in substr_map.items():
        mask = desc_lower.str.contains(k.lower(), na=False)
        tmp = df.loc[mask, ["vehicle_id","day"]].groupby(["vehicle_id","day"],as_index=False).size().rename(columns={"size":col})
        out = out.merge(tmp, on=["vehicle_id","day"], how="left")
    return out

def _episode_counts_from_mask(df, mask, gap_seconds, out_col):
    subset = df.loc[mask, ["vehicle_id","event_ts"]].sort_values(["vehicle_id","event_ts"]).copy()
    if subset.empty: return pd.DataFrame(columns=["vehicle_id","day",out_col])
    subset["prev_ts"] = subset.groupby("vehicle_id")["event_ts"].shift(1)
    subset["gap_s"] = (subset["event_ts"] - subset["prev_ts"]).dt.total_seconds()
    subset["new_episode"] = (subset["prev_ts"].isna() | (subset["gap_s"] > gap_seconds)).astype("int64")
    subset["episode_id"] = subset.groupby("vehicle_id")["new_episode"].cumsum()
    subset["day"] = subset["event_ts"].dt.floor("D")
    return (subset.groupby(["vehicle_id","day","episode_id"],as_index=False).size()
            .groupby(["vehicle_id","day"],as_index=False).size().rename(columns={"size":out_col}))

def _speeding_episode_breakdown(df, desc_lower):
    mask = _build_match_mask(desc_lower, SPEEDING_SUBSTRINGS)
    subset = df.loc[mask, ["vehicle_id","event_ts"]].sort_values(["vehicle_id","event_ts"]).copy()
    if subset.empty:
        return pd.DataFrame(columns=["vehicle_id","day","short_speeding_episode_count","long_speeding_episode_count","speeding_episode_count"])
    subset["prev_ts"] = subset.groupby("vehicle_id")["event_ts"].shift(1)
    subset["gap_s"] = (subset["event_ts"] - subset["prev_ts"]).dt.total_seconds()
    subset["new_episode"] = (subset["prev_ts"].isna() | (subset["gap_s"] > SPEEDING_EPISODE_WINDOW_SECONDS)).astype("int64")
    subset["episode_id"] = subset.groupby("vehicle_id")["new_episode"].cumsum()
    subset["day"] = subset["event_ts"].dt.floor("D")
    ep = subset.groupby(["vehicle_id","day","episode_id"],as_index=False).size().rename(columns={"size":"events_in_episode"})
    ep["short_speeding_episode_count"] = (ep["events_in_episode"] == 1).astype("int64")
    ep["long_speeding_episode_count"]  = (ep["events_in_episode"] > 1).astype("int64")
    ep["speeding_episode_count"] = 1
    return ep.groupby(["vehicle_id","day"],as_index=False)[["short_speeding_episode_count","long_speeding_episode_count","speeding_episode_count"]].sum()

def _aggregate(features, window):
    x = features.copy()
    if window == "month":
        x["window_start"] = _to_month_start(x["day"])
        x["window_end"]   = x["window_start"] + pd.offsets.MonthBegin(1)
    else:
        raise ValueError("window must be 'month'")
    out = x.groupby(["vehicle_id","window_start","window_end"],as_index=False).agg({c:"sum" for c in agg_sum_cols if c in x.columns})
    out["drive_hours"] = out["drive_seconds"] / 3600.0
    out["night_ratio"] = np.where(out["drive_seconds"] > 0, out["night_drive_seconds"] / out["drive_seconds"], 0.0)
    return out.sort_values(["vehicle_id","window_start"]).reset_index(drop=True)

def add_score_and_class(df_in, time_col):
    d = df_in.copy()
    if "window_end" not in d.columns:
        d["window_end"] = d[time_col] + pd.Timedelta(days=1)
    d["r_harsh_total"]      = _rank01(d["harsh_episode_count"])
    d["r_short_speeding"]   = _rank01(d["short_speeding_episode_count"])
    d["r_long_speeding"]    = _rank01(d["long_speeding_episode_count"])
    d["r_power_total"]      = _rank01(d["power_violation_episode_count"])
    d["r_camera_total"]     = _rank01(d["camera_obstruction_episode_count"])
    d["r_fatigue_total"]    = _rank01(d["fatigue_episode_count"])
    d["r_distraction_total"]= _rank01(d["driver_distraction_episode_count"])
    d["r_situational_total"]= _rank01(d["situational_risk_episode_count"])
    d["ubi_proxy_score"] = (
        0.20 * d["r_harsh_total"] + 0.12 * d["r_short_speeding"] + 0.18 * d["r_long_speeding"]
        + 0.10 * d["r_power_total"] + 0.05 * d["r_camera_total"] + 0.12 * d["r_fatigue_total"]
        + 0.13 * d["r_distraction_total"] + 0.10 * d["r_situational_total"]
    )
    d["insufficient_exposure_flag"] = (d["drive_hours"].fillna(0.0) < INSUFFICIENT_EXPOSURE_HOURS).astype("int64")
    ranked_mask = d["insufficient_exposure_flag"] == 0
    q1 = d.loc[ranked_mask, "ubi_proxy_score"].quantile(0.33) if ranked_mask.any() else 0.0
    q2 = d.loc[ranked_mask, "ubi_proxy_score"].quantile(0.67) if ranked_mask.any() else 0.0
    def lab(row):
        if row["insufficient_exposure_flag"] == 1: return "Insufficient Exposure"
        x = row["ubi_proxy_score"]
        if x <= q1: return "Low"
        if x <= q2: return "Medium"
        return "High"
    d["vehicle_behaviour_class"] = d.apply(lab, axis=1)
    keep = ["vehicle_id", time_col, "window_end", "drive_hours", "distance_km",
            "insufficient_exposure_flag", "harsh_episode_count", "short_speeding_episode_count",
            "long_speeding_episode_count", "speeding_episode_count", "power_violation_episode_count",
            "camera_obstruction_episode_count", "fatigue_episode_count", "driver_distraction_episode_count",
            "situational_risk_episode_count", "crash_count", "ubi_proxy_score", "vehicle_behaviour_class"]
    keep = [c for c in keep if c in d.columns]
    return d[keep].sort_values(["vehicle_id", time_col]).reset_index(drop=True)

def run_driver_ranking_pipeline(raw_df, show_output=True):
    df = raw_df.copy()
    df["event_ts"] = pd.to_datetime(df["event_ts"], utc=True, errors="coerce")
    df["event_description"] = df["event_description"].astype("string").str.strip()
    df["odometer"] = pd.to_numeric(df.get("odometer", np.nan), errors="coerce")
    df = df.dropna(subset=["vehicle_id","event_ts","event_description"]).copy()
    df["vehicle_id"] = df["vehicle_id"].astype("int64", errors="ignore")
    df = df.sort_values(["vehicle_id","event_ts"]).reset_index(drop=True)
    df["day"] = df["event_ts"].dt.floor("D")
    desc_lower = df["event_description"].astype("string").str.lower()

    v = df[["vehicle_id","event_ts","odometer"]].copy()
    v["next_ts"]  = v.groupby("vehicle_id")["event_ts"].shift(-1)
    v["next_odo"] = v.groupby("vehicle_id")["odometer"].shift(-1)
    v["delta_s"]  = (v["next_ts"] - v["event_ts"]).dt.total_seconds()
    intervals = v.loc[v["delta_s"].notna() & (v["delta_s"] > 0) & (v["delta_s"] <= SESSION_GAP_SECONDS)].copy()
    intervals = intervals.rename(columns={"event_ts":"t0","next_ts":"t1","odometer":"odo0","next_odo":"odo1"})
    intervals["delta_km"] = np.where(intervals["odo0"].notna() & intervals["odo1"].notna(), intervals["odo1"] - intervals["odo0"], np.nan)
    intervals.loc[intervals["delta_km"] < 0, "delta_km"] = np.nan

    rows = []
    for r in intervals.itertuples(index=False):
        parts = _split_interval_by_day(r.t0, r.t1)
        if not parts: continue
        total_s = (r.t1 - r.t0).total_seconds()
        if total_s <= 0: continue
        for day_start, p0, p1 in parts:
            dur_s   = (p1 - p0).total_seconds()
            night_s = _night_overlap_seconds(p0, p1)
            km_part = float(r.delta_km) * (dur_s / total_s) if pd.notna(r.delta_km) else np.nan
            night_km_part = (km_part * (night_s / dur_s)) if (dur_s > 0 and pd.notna(km_part)) else np.nan
            rows.append((r.vehicle_id, day_start, dur_s, night_s, km_part, night_km_part))

    drive_daily = pd.DataFrame(rows, columns=["vehicle_id","day","drive_seconds","night_drive_seconds","distance_km_part","night_distance_km_part"])
    if drive_daily.empty:
        drive_daily = pd.DataFrame(columns=["vehicle_id","day","drive_seconds","night_drive_seconds","distance_km_part","night_distance_km_part"])
    drive_daily = drive_daily.groupby(["vehicle_id","day"],as_index=False).agg(
        drive_seconds=("drive_seconds","sum"), night_drive_seconds=("night_drive_seconds","sum"),
        distance_km=("distance_km_part","sum"), night_distance_km=("night_distance_km_part","sum"))
    drive_daily["drive_hours"] = drive_daily["drive_seconds"] / 3600.0
    drive_daily["night_ratio"] = np.where(drive_daily["drive_seconds"] > 0, drive_daily["night_drive_seconds"] / drive_daily["drive_seconds"], 0.0)

    counts_daily = df[["vehicle_id","day"]].drop_duplicates().copy()
    for substr_map in [HARSH_SUBSTRINGS, SPEEDING_SUBSTRINGS, POWER_SUBSTRINGS, CAMERA_SUBSTRINGS,
                       FATIGUE_SUBSTRINGS, DISTRACTION_SUBSTRINGS, SITUATIONAL_SUBSTRINGS, CRASH_SUBSTRINGS]:
        counts_daily = _add_raw_counts(df, desc_lower, counts_daily, substr_map)
    for _, col in ALL_EVENT_COUNT_MAPS:
        if col not in counts_daily.columns: counts_daily[col] = 0
        counts_daily[col] = counts_daily[col].fillna(0).astype("int64")

    harsh_ep        = _episode_counts_from_mask(df, _build_match_mask(desc_lower, HARSH_SUBSTRINGS),        HARSH_EPISODE_WINDOW_SECONDS,       "harsh_episode_count")
    speeding_ep     = _speeding_episode_breakdown(df, desc_lower)
    power_ep        = _episode_counts_from_mask(df, _build_match_mask(desc_lower, POWER_SUBSTRINGS),        POWER_EPISODE_WINDOW_SECONDS,       "power_violation_episode_count")
    camera_ep       = _episode_counts_from_mask(df, _build_match_mask(desc_lower, CAMERA_SUBSTRINGS),       CAMERA_EPISODE_WINDOW_SECONDS,      "camera_obstruction_episode_count")
    fatigue_ep      = _episode_counts_from_mask(df, _build_match_mask(desc_lower, FATIGUE_SUBSTRINGS),      FATIGUE_EPISODE_WINDOW_SECONDS,     "fatigue_episode_count")
    distraction_ep  = _episode_counts_from_mask(df, _build_match_mask(desc_lower, DISTRACTION_SUBSTRINGS),  DISTRACTION_EPISODE_WINDOW_SECONDS, "driver_distraction_episode_count")
    situational_ep  = _episode_counts_from_mask(df, _build_match_mask(desc_lower, SITUATIONAL_SUBSTRINGS),  SITUATIONAL_EPISODE_WINDOW_SECONDS, "situational_risk_episode_count")

    features_daily = counts_daily.merge(drive_daily, on=["vehicle_id","day"], how="left")
    for ep_df in [harsh_ep, speeding_ep, power_ep, camera_ep, fatigue_ep, distraction_ep, situational_ep]:
        features_daily = features_daily.merge(ep_df, on=["vehicle_id","day"], how="left")

    fill0 = {"drive_seconds":0.0,"night_drive_seconds":0.0,"drive_hours":0.0,"night_ratio":0.0,
             "harsh_episode_count":0,"short_speeding_episode_count":0,"long_speeding_episode_count":0,
             "speeding_episode_count":0,"power_violation_episode_count":0,"camera_obstruction_episode_count":0,
             "fatigue_episode_count":0,"driver_distraction_episode_count":0,"situational_risk_episode_count":0}
    features_daily = features_daily.fillna(fill0)
    features_daily["harsh_total_count"]              = features_daily["harsh_corner_count"] + features_daily["harsh_braking_count"] + features_daily["harsh_accel_count"]
    features_daily["fatigue_total_count"]            = features_daily["eye_closed_count"] + features_daily["yawn_count"] + features_daily["fatigue_count"]
    features_daily["driver_distraction_total_count"] = features_daily["distraction_count"] + features_daily["phone_use_count"] + features_daily["smoke_count"] + features_daily["Driver_Seatbelt_not_on"]
    features_daily["situational_risk_total_count"]   = features_daily["headway_monitor_count"] + features_daily["lane_departure_count"] + features_daily["forward_collision_count"] + features_daily["ped_collision_count"]
    features_daily = features_daily.sort_values(["vehicle_id","day"]).reset_index(drop=True)

    features_monthly = _aggregate(features_daily, "month")
    features_daily["window_start"] = features_daily["day"]
    features_daily["window_end"]   = features_daily["day"] + pd.Timedelta(days=1)

    scored_monthly_out = add_score_and_class(features_monthly, "window_start")

    return {
        "df": df,
        "drive_daily": drive_daily,
        "features_daily": features_daily,
        "features_monthly": features_monthly,
        "scored_monthly": scored_monthly_out,
    }

print("✓ Ranking pipeline loaded")

# BUILD RAW_DF_VLM + RANKINGS

In [ ]:
# =============================================================================
# CELL 4 — BUILD RAW_DF_VLM + EVENT AUDIT + VLM RANKINGS
# =============================================================================
# Injects VLM detections into raw_df to produce raw_df_vlm.
# Runs ranking pipeline on both and compares results.
# =============================================================================

def _safe_str(x) -> str:
    if x is None: return ""
    try:
        if pd.isna(x): return ""
    except (TypeError, ValueError): pass
    s = str(x).strip()
    return "" if s in ("<NA>", "nan", "None") else s

def _events_for_pipeline_from_row(row):
    det_events = row.get("det_rankable_event_descriptions", [])
    rev_events = row.get("rev_rankable_event_descriptions", [])
    if not isinstance(det_events, list): det_events = []
    if not isinstance(rev_events, list): rev_events = []
    events = det_events + rev_events if INCLUDE_REVIEW_EVENTS_IN_RANKING else det_events
    return list(dict.fromkeys([e for e in events if e in RANKABLE_VLM_EVENTS_FOR_PIPELINE]))

def build_raw_df_vlm_and_audit(raw_df, fused_event_df):
    raw_base = raw_df.copy()
    raw_base["vehicle_id"] = pd.to_numeric(raw_base["vehicle_id"], errors="coerce")
    raw_base["terminal_event_id"] = raw_base["terminal_event_id"].astype(str).str.strip()
    raw_base["terminal_event_id"] = raw_base["terminal_event_id"].where(
        ~raw_base["terminal_event_id"].isin(["nan","None","NaT",""]), other=None)
    raw_base["event_ts"] = pd.to_datetime(raw_base["event_ts"], utc=True, errors="coerce")
    raw_base["event_description"] = raw_base["event_description"].astype("string").str.strip()

    fused = fused_event_df.copy()
    fused["vehicle_id"] = pd.to_numeric(fused["vehicle_id"], errors="coerce")
    fused["terminal_event_id"] = fused["terminal_event_id"].astype(str).str.strip()
    fused["terminal_event_id"] = fused["terminal_event_id"].where(
        ~fused["terminal_event_id"].isin(["nan","None","NaT",""]), other=None)
    fused["event_ts_final"] = pd.to_datetime(fused["event_ts_final"], utc=True, errors="coerce")
    fused["original_event_description"] = fused["original_event_description"].astype("string").str.strip()
    fused["vlm_events_for_pipeline"] = fused.apply(_events_for_pipeline_from_row, axis=1)

    base_cols = list(raw_base.columns)
    rows_to_add, rows_to_remove, audit_rows = [], [], []

    def _make_new_row(vehicle_id, terminal_event_id, event_ts_final, ev, raw_match):
        new_row = {c: np.nan for c in base_cols}
        new_row["vehicle_id"]        = vehicle_id
        new_row["terminal_event_id"] = terminal_event_id
        new_row["event_ts"]          = event_ts_final
        new_row["event_description"] = ev
        if len(raw_match) > 0:
            first = raw_match.iloc[0]
            for c in base_cols:
                if c not in ["vehicle_id","terminal_event_id","event_ts","event_description"]:
                    new_row[c] = first[c]
        return new_row

    def _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
               action, reason, removed="", added="", changed=1):
        audit_rows.append({
            "clip": clip, "vehicle_id": vehicle_id, "terminal_event_id": terminal_event_id,
            "original_event_description": original_event, "vlm_events_for_pipeline": vlm_events,
            "vlm_change_type": change_type, "change_action": action, "change_reason": reason,
            "removed_event": removed, "added_event": added, "event_changed_flag": changed,
        })

    for r in fused.itertuples(index=False):
        clip              = getattr(r, "clip", np.nan)
        vehicle_id        = getattr(r, "vehicle_id", np.nan)
        terminal_event_id = getattr(r, "terminal_event_id", None)
        event_ts_final    = getattr(r, "event_ts_final", pd.NaT)
        original_event    = _safe_str(getattr(r, "original_event_description", ""))
        vlm_events        = getattr(r, "vlm_events_for_pipeline", [])
        change_type       = _safe_str(getattr(r, "vlm_change_type", ""))

        if pd.isna(vehicle_id) or terminal_event_id is None or str(terminal_event_id).strip() in ["nan","None","NaT",""]:
            _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                   "no_change", "missing vehicle_id or terminal_event_id", changed=0)
            continue

        raw_match = raw_base.loc[
            (raw_base["vehicle_id"] == vehicle_id) &
            (raw_base["terminal_event_id"] == str(terminal_event_id))
        ].copy()

        raw_events_here = sorted(raw_match["event_description"].dropna().astype("string").str.strip().unique().tolist())
        original_exists_in_raw = original_event in raw_events_here if original_event != "" else False
        if not isinstance(vlm_events, list): vlm_events = []

        if change_type == "no_vlm_signal":
            _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                   "no_change", "VLM produced no usable event", changed=0)
            continue

        if change_type == "confirmed":
            added_any = False
            for ev in vlm_events:
                if ev not in raw_events_here:
                    rows_to_add.append(_make_new_row(vehicle_id, terminal_event_id, event_ts_final, ev, raw_match))
                    _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                           "added", "original event stayed, extra VLM event added", added=ev)
                    added_any = True
            if not added_any:
                _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                       "no_change", "original event stayed, no extra VLM event needed", changed=0)
            continue

        if change_type in ("not_confirmed", "contradicted"):
            removed_any = added_any = False
            reason_remove = ("original event not confirmed by VLM, so it was removed"
                             if change_type == "not_confirmed"
                             else "VLM contradicted original telematics event — original removed")
            reason_add = ("replacement VLM event added after original was not confirmed"
                          if change_type == "not_confirmed"
                          else "VLM contradicted original — replacement VLM event added")
            if original_exists_in_raw:
                rows_to_remove.append({"vehicle_id": vehicle_id, "terminal_event_id": terminal_event_id, "event_description": original_event})
                _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                       "removed", reason_remove, removed=original_event)
                removed_any = True
            for ev in vlm_events:
                if ev not in raw_events_here:
                    rows_to_add.append(_make_new_row(vehicle_id, terminal_event_id, event_ts_final, ev, raw_match))
                    _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                           "added", reason_add, added=ev)
                    added_any = True
            if not removed_any and not added_any:
                _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                       "no_change", "original not in raw_df and no replacement available", changed=0)
            continue

        added_any = False
        for ev in vlm_events:
            if ev not in raw_events_here:
                rows_to_add.append(_make_new_row(vehicle_id, terminal_event_id, event_ts_final, ev, raw_match))
                _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                       "added", f"{change_type} added extra VLM context", added=ev)
                added_any = True
        if not added_any:
            _audit(clip, vehicle_id, terminal_event_id, original_event, vlm_events, change_type,
                   "no_change", f"{change_type} found nothing new to add", changed=0)

    raw_df_vlm = raw_base.copy()

    if rows_to_remove:
        remove_df = pd.DataFrame(rows_to_remove).drop_duplicates().copy()
        remove_df["key"] = remove_df["vehicle_id"].astype(str) + "|" + remove_df["terminal_event_id"].astype(str) + "|" + remove_df["event_description"].astype(str)
        raw_df_vlm["key"] = raw_df_vlm["vehicle_id"].astype(str) + "|" + raw_df_vlm["terminal_event_id"].astype(str) + "|" + raw_df_vlm["event_description"].astype(str)
        raw_df_vlm = raw_df_vlm.loc[~raw_df_vlm["key"].isin(remove_df["key"])].copy()
        raw_df_vlm = raw_df_vlm.drop(columns=["key"])

    if rows_to_add:
        raw_df_vlm = pd.concat([raw_df_vlm, pd.DataFrame(rows_to_add, columns=base_cols)], ignore_index=True, sort=False)

    raw_df_vlm["event_ts"] = pd.to_datetime(raw_df_vlm["event_ts"], utc=True, errors="coerce")
    raw_df_vlm["event_description"] = raw_df_vlm["event_description"].astype("string").str.strip()
    raw_df_vlm = raw_df_vlm.sort_values(["vehicle_id","event_ts","terminal_event_id","event_description"]).reset_index(drop=True)

    return raw_df_vlm, pd.DataFrame(audit_rows)

def compare_scored_outputs(base_scored, vlm_scored, time_col):
    base = base_scored.copy().rename(columns={"ubi_proxy_score":"ubi_proxy_score_base","vehicle_behaviour_class":"vehicle_behaviour_class_base"})
    vlm  = vlm_scored.copy().rename(columns={"ubi_proxy_score":"ubi_proxy_score_vlm","vehicle_behaviour_class":"vehicle_behaviour_class_vlm"})
    base["rank_base"] = base.groupby(time_col)["ubi_proxy_score_base"].rank(method="dense", ascending=False)
    vlm["rank_vlm"]   = vlm.groupby(time_col)["ubi_proxy_score_vlm"].rank(method="dense", ascending=False)
    vlm_merge_cols = ["vehicle_id", time_col, "ubi_proxy_score_vlm", "vehicle_behaviour_class_vlm", "rank_vlm", "covered_ratio", "covered_clips", "total_clips"]
    vlm_merge_cols = [c for c in vlm_merge_cols if c in vlm.columns]
    out = base.merge(vlm[vlm_merge_cols], on=["vehicle_id", time_col], how="outer")
    out["score_delta"] = out["ubi_proxy_score_vlm"].fillna(0) - out["ubi_proxy_score_base"].fillna(0)
    out["rank_delta"]  = out["rank_base"].fillna(0) - out["rank_vlm"].fillna(0)
    return out.sort_values([time_col, "rank_vlm", "vehicle_id"]).reset_index(drop=True)

# ── Build adjusted raw table + audit ─────────────────────────────────────────
raw_df_vlm, event_change_audit_df = build_raw_df_vlm_and_audit(raw_df, fused_event_df)

print(f"raw_df rows              : {len(raw_df)}")
print(f"raw_df_vlm rows          : {len(raw_df_vlm)}")
added_rows = raw_df_vlm.iloc[len(raw_df):]
print(f"Added rows               : {len(added_rows)}")
print(f"  with valid event_ts    : {added_rows['event_ts'].notna().sum()}")

# ── Covered ratio ─────────────────────────────────────────────────────────────
total_clips_per_vehicle   = fused_event_df.groupby("vehicle_id").size().reset_index(name="total_clips")
covered_clips_per_vehicle = fused_event_df[fused_event_df["cam_covered_flag"] == 1].groupby("vehicle_id").size().reset_index(name="covered_clips")
covered_ratio_df = total_clips_per_vehicle.merge(covered_clips_per_vehicle, on="vehicle_id", how="left").fillna({"covered_clips": 0})
covered_ratio_df["covered_ratio"] = (covered_ratio_df["covered_clips"] / covered_ratio_df["total_clips"]).round(3)

# ── Run VLM-adjusted rankings ─────────────────────────────────────────────────
vlm_outputs        = run_driver_ranking_pipeline(raw_df_vlm, show_output=False)
scored_monthly_vlm = vlm_outputs["scored_monthly"]
scored_monthly_vlm = scored_monthly_vlm.merge(
    covered_ratio_df[["vehicle_id","covered_ratio","total_clips","covered_clips"]],
    on="vehicle_id", how="left"
).fillna({"covered_ratio": 0.0, "covered_clips": 0})

# ── Compare ───────────────────────────────────────────────────────────────────
monthly_rank_compare_df = compare_scored_outputs(scored_monthly, scored_monthly_vlm, "window_start")

final_cols = ["vehicle_id","window_start","ubi_proxy_score_base","ubi_proxy_score_vlm","score_delta",
              "rank_base","rank_vlm","rank_delta","vehicle_behaviour_class_base","vehicle_behaviour_class_vlm",
              "covered_ratio","covered_clips","total_clips"]
final_cols = [c for c in final_cols if c in monthly_rank_compare_df.columns]
final_monthly_rankings_df = (
    monthly_rank_compare_df[final_cols]
    .sort_values(["window_start","rank_vlm","vehicle_id"])
    .reset_index(drop=True)
)

changed_vehicle_ids = (
    event_change_audit_df.loc[event_change_audit_df["event_changed_flag"] == 1, "vehicle_id"]
    .dropna().astype("int64").unique().tolist()
)

print(f"\nNon-zero score deltas    : {(monthly_rank_compare_df['score_delta'] != 0).sum()}")
print(f"Max score delta          : {monthly_rank_compare_df['score_delta'].max():.4f}")
print(f"Min score delta          : {monthly_rank_compare_df['score_delta'].min():.4f}")
print(f"Changed vehicles         : {len(changed_vehicle_ids)}")
print("\n[Changed vehicles — base vs VLM ranking]")
impact_cols = ["vehicle_id","window_start","ubi_proxy_score_base","ubi_proxy_score_vlm","score_delta",
               "rank_base","rank_vlm","rank_delta","vehicle_behaviour_class_base","vehicle_behaviour_class_vlm"]
impact_cols = [c for c in impact_cols if c in monthly_rank_compare_df.columns]
display(
    monthly_rank_compare_df
    .loc[monthly_rank_compare_df["vehicle_id"].isin(changed_vehicle_ids), impact_cols]
    .sort_values("score_delta", ascending=False)
    .reset_index(drop=True)
)
print("\n[Full final monthly rankings]")
display(final_monthly_rankings_df)

# Save docs for LLM

In [ ]:
# =============================================================================
# SAVE OUTPUTS FOR LLM QUERY NOTEBOOK
# =============================================================================
# Run after Cell 4 completes.
# Saves everything needed for the LLM query layer — VLM-adjusted rankings
# and all supporting data. No original telematics-only outputs saved.
# =============================================================================

import os
import pandas as pd

OUTPUTS_DIR = "outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

def _safe_list_col(df, col):
    """Convert list columns to string for CSV export."""
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: str(x) if isinstance(x, list) else x
        )
    return df

# ── 1. VLM-adjusted final monthly rankings ───────────────────────────────────
final_monthly_rankings_df.to_csv(
    os.path.join(OUTPUTS_DIR, "final_monthly_rankings_vlm.csv"),
    index=False, encoding="utf-8"
)

# ── 2. Full scored monthly VLM (includes all episode counts) ─────────────────
scored_monthly_vlm.to_csv(
    os.path.join(OUTPUTS_DIR, "scored_monthly_vlm.csv"),
    index=False, encoding="utf-8"
)

# ── 3. Event change audit — what VLM added/removed per clip ──────────────────
audit_to_save = event_change_audit_df.copy()
audit_to_save = _safe_list_col(audit_to_save, "vlm_events_for_pipeline")
audit_to_save.to_csv(
    os.path.join(OUTPUTS_DIR, "event_change_audit.csv"),
    index=False, encoding="utf-8"
)

# ── 4. Fused event df — VLM detections per clip with vehicle linkage ──────────
fused_to_save = fused_event_df.copy()
for col in ["det_codes", "rev_codes", "det_event_descriptions", "rev_event_descriptions",
            "det_rankable_event_descriptions", "rev_rankable_event_descriptions",
            "extra_rankable_vlm_events"]:
    fused_to_save = _safe_list_col(fused_to_save, col)

# Keep terminal_event_id as string
if "terminal_event_id" in fused_to_save.columns:
    fused_to_save["terminal_event_id"] = (
        fused_to_save["terminal_event_id"]
        .astype(str).str.strip()
    )
    fused_to_save["terminal_event_id"] = fused_to_save["terminal_event_id"].where(
        ~fused_to_save["terminal_event_id"].isin(["nan", "None", "NaT", ""]),
        other=None
    )

fused_to_save.to_csv(
    os.path.join(OUTPUTS_DIR, "fused_event_df.csv"),
    index=False, encoding="utf-8"
)

# ── 5. Covered ratio per vehicle ──────────────────────────────────────────────
covered_ratio_df.to_csv(
    os.path.join(OUTPUTS_DIR, "covered_ratio.csv"),
    index=False, encoding="utf-8"
)

# ── Verify ────────────────────────────────────────────────────────────────────
print("Saved to outputs/:")
for f in sorted(os.listdir(OUTPUTS_DIR)):
    path = os.path.join(OUTPUTS_DIR, f)
    size = os.path.getsize(path) / 1024 / 1024
    print(f"  {f:<45} {size:.1f} MB")

print(f"\nfinal_monthly_rankings_vlm : {final_monthly_rankings_df.shape}")
print(f"scored_monthly_vlm         : {scored_monthly_vlm.shape}")
print(f"event_change_audit         : {event_change_audit_df.shape}")
print(f"fused_event_df             : {fused_to_save.shape}")
print(f"covered_ratio              : {covered_ratio_df.shape}")

# had to add this so i can rerun fleets
# ── 6. Features monthly VLM — needed for per-fleet re-ranking ─────────────────
vlm_outputs["features_monthly"].to_csv(
    os.path.join(OUTPUTS_DIR, "features_monthly_vlm.csv"),
    index=False, encoding="utf-8"
)
print(f"features_monthly_vlm       : {vlm_outputs['features_monthly'].shape}")

# SIGNAL DIAGNOSTICS

In [ ]:
# =============================================================================
# CELL 5 — SIGNAL DIAGNOSTICS
# =============================================================================
# Sanity checks on VLM signal quality and pipeline outputs.
# =============================================================================

print("=== WHY no_vlm_signal ===\n")

# ── Tier breakdown ────────────────────────────────────────────────────────────
tier1_clips = fused_event_df[
    (fused_event_df["cam_s"] == 2) &
    (fused_event_df["cam_covered_flag"] == 1)
]
tier2_clips = fused_event_df[
    fused_event_df["rev_codes"].apply(
        lambda x: 67 in x if isinstance(x, list) else False
    )
]

gated     = fused_event_df[fused_event_df["cam_s"].isin([2, 3, 4])]
not_gated = fused_event_df[~fused_event_df["cam_s"].isin([2, 3, 4])]

print(f"\nCamera quality tier breakdown:")
print(f"  Tier 1 — definitely covered (VLM skipped, det=[6]) : {len(tier1_clips)}")
print(f"  Tier 2 — possible covering  (VLM ran, rev=[67])    : {len(tier2_clips)}")
print(f"  Tier 0 — camera OK          (VLM ran normally)     : {len(fused_event_df) - len(tier1_clips) - len(tier2_clips)}")
print(f"\nNote: Original pipeline incorrectly marked 675 clips as camera covered.")
print(f"After threshold correction, only {len(tier1_clips)} are confirmed covered.")
print(f"The remaining ~{675 - len(tier1_clips)} clips now have VLM analysis for the first time.")

print(f"\nKey finding:")
print(f"  {len(tier1_clips)} clips confirmed camera covered (tier 1 — solid obstruction)")
print(f"  {len(tier2_clips)} clips possible covering under review (tier 2 — cream/beads/blur)")

no_signal_not_gated = not_gated[not_gated["vlm_change_type"] == "no_vlm_signal"]
new_context_not_gated = not_gated[not_gated["vlm_change_type"] == "new_context_added"]

print(f"  {len(new_context_not_gated)} non-gated clips had VLM detect new behaviour with no prior telematics label")
print(f"    These represent events the AI camera system missed entirely")
print(f"  {len(no_signal_not_gated)} non-gated clips had VLM run but find nothing — no change made")

print("\nChange type counts (non-gated clips only):")
display(
    not_gated["vlm_change_type"]
    .value_counts(dropna=False)
    .rename_axis("vlm_change_type")
    .reset_index(name="count")
)

print("\nPrimary VLM event distribution (all clips):")
display(
    fused_event_df["primary_vlm_event"]
    .replace("", "nothing_detected")
    .value_counts(dropna=False)
    .rename_axis("primary_vlm_event")
    .reset_index(name="count")
)

print("\ncam_s distribution:")
print(fused_event_df["cam_s"].value_counts(dropna=False))

print(f"\nfused_event_df shape     : {fused_event_df.shape}")
print(f"Unique clips             : {fused_event_df['clip'].nunique()}")
rows_per_clip = fused_event_df.groupby("clip").size()
print("\nRows per clip distribution:")
print(rows_per_clip.value_counts().head(10))

print(f"\nraw_df rows              : {len(raw_df)}")
print(f"raw_df_vlm rows          : {len(raw_df_vlm)}")
print(f"\nNon-zero score deltas    : {(monthly_rank_compare_df['score_delta'] != 0).sum()}")
print(f"Max score delta          : {monthly_rank_compare_df['score_delta'].max():.4f}")
print(f"Min score delta          : {monthly_rank_compare_df['score_delta'].min():.4f}")

# CASE STUDY: TOP SEATBELT VEHICLE

In [ ]:
# =============================================================================
# CELL 6 — CASE STUDY: TOP SEATBELT VEHICLE
# =============================================================================
# Identifies the vehicle with the most SEATBELT_D_OFF events added by VLM
# and traces the full impact from event injection through to rank change.
# =============================================================================

seatbelt_added = (
    event_change_audit_df[
        (event_change_audit_df["change_action"] == "added") &
        (event_change_audit_df["added_event"] == "SEATBELT_D_OFF")
    ]
    .groupby("vehicle_id").size().sort_values(ascending=False)
)
print("Top 10 vehicles by SEATBELT_D_OFF events added by VLM:")
print(seatbelt_added.head(10))

top_vehicle = seatbelt_added.index[0]
print(f"\nFocusing on vehicle_id: {top_vehicle}")

before = raw_df[
    (raw_df["vehicle_id"] == top_vehicle) &
    (raw_df["event_description"] == "SEATBELT_D_OFF")
][["vehicle_id","terminal_event_id","event_description","event_ts"]].sort_values("event_ts")
print(f"\nSEATBELT_D_OFF BEFORE VLM: {len(before)} events")
display(before.head(20))

after = raw_df_vlm[
    (raw_df_vlm["vehicle_id"] == top_vehicle) &
    (raw_df_vlm["event_description"] == "SEATBELT_D_OFF")
][["vehicle_id","terminal_event_id","event_description","event_ts"]].sort_values("event_ts")
print(f"\nSEATBELT_D_OFF AFTER VLM: {len(after)} events")
display(after.head(20))

single_vehicle_base = run_driver_ranking_pipeline(raw_df[raw_df["vehicle_id"] == top_vehicle].copy(), show_output=False)
single_vehicle_vlm  = run_driver_ranking_pipeline(raw_df_vlm[raw_df_vlm["vehicle_id"] == top_vehicle].copy(), show_output=False)

ep_before = single_vehicle_base["features_monthly"]["driver_distraction_episode_count"].sum()
ep_after  = single_vehicle_vlm["features_monthly"]["driver_distraction_episode_count"].sum()
score_row = monthly_rank_compare_df[monthly_rank_compare_df["vehicle_id"] == top_vehicle].iloc[0]

print(f"\nCASE STUDY — Vehicle {top_vehicle}")
print("=" * 60)

print("\nEvents added by VLM:")
vehicle_added = event_change_audit_df[
    (event_change_audit_df["vehicle_id"] == top_vehicle) &
    (event_change_audit_df["change_action"] == "added")
]["added_event"].value_counts()

total_added = cam_covered_added = 0
for event, count in vehicle_added.items():
    b = raw_df[(raw_df["vehicle_id"] == top_vehicle) & (raw_df["event_description"] == event)].shape[0]
    a = raw_df_vlm[(raw_df_vlm["vehicle_id"] == top_vehicle) & (raw_df_vlm["event_description"] == event)].shape[0]
    print(f"  {event:25s}: {b} → {a}  (+{count})")
    total_added += count
    if event == "v_cam_covered": cam_covered_added += count
print(f"  {'TOTAL':25s}: +{total_added} new events")

ep_rankable = total_added - cam_covered_added
print(f"\nDistraction episodes:")
print(f"  Before : {ep_before:.0f}")
print(f"  After  : {ep_after:.0f}  (+{ep_after - ep_before:.0f})")

print(f"\nUBI Score:")
print(f"  Before : {score_row['ubi_proxy_score_base']:.4f}")
print(f"  After  : {score_row['ubi_proxy_score_vlm']:.4f}  ({score_row['score_delta']:+.4f})")

rank_direction = "worse" if score_row["rank_delta"] > 0 else "better"
print(f"\nRank (out of {monthly_rank_compare_df['vehicle_id'].nunique()} vehicles):")
print(f"  Before : {score_row['rank_base']:.0f}")
print(f"  After  : {score_row['rank_vlm']:.0f}  ({score_row['rank_delta']:+.0f} — {rank_direction})")

print(f"\nBehaviour Class:")
print(f"  Before : {score_row['vehicle_behaviour_class_base']}")
print(f"  After  : {score_row['vehicle_behaviour_class_vlm']}")

fleet_ep = scored_monthly_vlm["driver_distraction_episode_count"].dropna()
pct = (fleet_ep < ep_after).mean() * 100
print(f"\nFleet distraction episodes — mean={fleet_ep.mean():.0f}  max={fleet_ep.max():.0f}  median={fleet_ep.median():.0f}")
print(f"This vehicle sits at the {pct:.0f}th percentile of the fleet.")

# ANALYTICS 1: FLEET OVERVIEW

In [ ]:
# =============================================================================
# ANALYTICS 1 — FLEET-LEVEL OVERVIEW
# Purpose: Show how many vehicles changed class/rank after VLM injection.
# Thesis point: Quantifies the incremental value of vision-based analysis.
# =============================================================================

print("=" * 70)
print("ANALYTICS 1 — FLEET-LEVEL OVERVIEW")
print("=" * 70)

total_vehicles = monthly_rank_compare_df["vehicle_id"].nunique()
changed        = (monthly_rank_compare_df["score_delta"] != 0).sum()
rank_improved  = (monthly_rank_compare_df["rank_delta"] < 0).sum()
rank_worsened  = (monthly_rank_compare_df["rank_delta"] > 0).sum()
rank_unchanged = (monthly_rank_compare_df["rank_delta"] == 0).sum()

print(f"Total vehicles in ranking              : {total_vehicles}")
print(f"Vehicles with score change             : {changed} ({100*changed/len(monthly_rank_compare_df):.1f}%)")
print(f"Rank improved (VLM found fewer violations): {rank_improved}")
print(f"Rank worsened (VLM found more violations) : {rank_worsened}")
print(f"Rank unchanged                         : {rank_unchanged}")

class_change = monthly_rank_compare_df[
    monthly_rank_compare_df["vehicle_behaviour_class_base"] !=
    monthly_rank_compare_df["vehicle_behaviour_class_vlm"]
]
print(f"\nVehicles that changed behaviour class  : {len(class_change)}")
if len(class_change) > 0:
    display(
        class_change.groupby(["vehicle_behaviour_class_base","vehicle_behaviour_class_vlm"])
        .size().reset_index(name="count").sort_values("count", ascending=False)
    )

# ANALYTICS 2: SCORE DELTA DISTRIBUTION

In [ ]:
# =============================================================================
# ANALYTICS 2 — SCORE DELTA DISTRIBUTION
# Purpose: Show the distribution of score changes across the fleet.
# Thesis point: Most vehicles are unaffected; a tail of high-risk drivers
#               are exposed by VLM that telematics alone missed.
# =============================================================================

print("=" * 70)
print("ANALYTICS 2 — SCORE DELTA DISTRIBUTION")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
deltas = monthly_rank_compare_df["score_delta"].dropna()
ax.hist(deltas[deltas != 0], bins=30, color=C_DIFF, edgecolor="white", alpha=0.85)
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_title("Score Delta Distribution\n(non-zero changes only)", fontsize=11)
ax.set_xlabel("Score Delta (VLM − Baseline)")
ax.set_ylabel("Number of Vehicles")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

ax = axes[1]
rank_deltas = monthly_rank_compare_df["rank_delta"].dropna()
ax.hist(rank_deltas[rank_deltas != 0], bins=20, color=C_VLM, edgecolor="white", alpha=0.85)
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_title("Rank Delta Distribution\n(non-zero changes only)", fontsize=11)
ax.set_xlabel("Rank Delta (positive = improved rank)")
ax.set_ylabel("Number of Vehicles")

plt.tight_layout()
plt.show()

print(f"\nScore delta summary:")
print(deltas.describe().round(4))

print("\nVehicle that moved UP most (VLM found most new violations):")
display(
    monthly_rank_compare_df.sort_values("rank_delta", ascending=False).head(1)
    [["vehicle_id","window_start","rank_base","rank_vlm","rank_delta",
      "ubi_proxy_score_base","ubi_proxy_score_vlm","score_delta",
      "vehicle_behaviour_class_base","vehicle_behaviour_class_vlm"]]
)

worst_mover = monthly_rank_compare_df.sort_values("rank_delta", ascending=False).iloc[0]["vehicle_id"]
print("\nAll audit rows for biggest UP mover:")
display(
    event_change_audit_df[
        (event_change_audit_df["vehicle_id"] == worst_mover) &
        (event_change_audit_df["event_changed_flag"] == 1)
    ][["clip","original_event_description","vlm_change_type","change_action","removed_event","added_event"]]
    .reset_index(drop=True)
)

print("\nVehicle that moved DOWN most (VLM confirmed fewer violations):")
display(
    monthly_rank_compare_df.sort_values("rank_delta", ascending=True).head(1)
    [["vehicle_id","window_start","rank_base","rank_vlm","rank_delta",
      "ubi_proxy_score_base","ubi_proxy_score_vlm","score_delta",
      "vehicle_behaviour_class_base","vehicle_behaviour_class_vlm"]]
)

best_mover = monthly_rank_compare_df.sort_values("rank_delta", ascending=True).iloc[0]["vehicle_id"]
print("\nAll audit rows for biggest DOWN mover:")
display(
    event_change_audit_df[
        (event_change_audit_df["vehicle_id"] == best_mover) &
        (event_change_audit_df["event_changed_flag"] == 1)
    ][["clip","original_event_description","vlm_change_type","change_action","removed_event","added_event"]]
    .reset_index(drop=True)
)

# ANALYTICS 3: VLM DETECTION BREAKDOWN

In [ ]:
# =============================================================================
# ANALYTICS 3 — VLM DETECTION BREAKDOWN
# Purpose: Show what the VLM detected across all clips.
# Thesis point: Demonstrates the breadth of VLM coverage across behaviour types.
# =============================================================================

print("=" * 70)
print("ANALYTICS 3 — VLM DETECTION BREAKDOWN")
print("=" * 70)

detection_counts = {
    "Phone use"      : fused_event_df["phone_flag"].sum(),
    "Seatbelt off"   : fused_event_df["seatbelt_off_flag"].sum(),
    "Camera covered" : fused_event_df["cam_covered_flag"].sum(),
    "Fatigue"        : fused_event_df["fatigue_flag"].sum(),
    "Distraction"    : fused_event_df["distraction_flag"].sum(),
    "Smoking"        : fused_event_df["smoke_flag"].sum(),
    "Face obstructed": fused_event_df["face_obstructed_review_flag"].sum(),
}

det_df = pd.DataFrame(detection_counts.items(), columns=["Behaviour","VLM Detections"]).sort_values("VLM Detections", ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(det_df["Behaviour"], det_df["VLM Detections"], color=C_VLM, edgecolor="white")
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_title("VLM Detections by Behaviour Type\n(across all processed clips)", fontsize=11)
ax.set_xlabel("Number of Clips")
plt.tight_layout()
plt.show()

display(det_df.sort_values("VLM Detections", ascending=False))

# ANALYTICS 4: VLM vs AI Camera AGREEMENT

In [ ]:
# =============================================================================
# ANALYTICS 4 — VLM vs AI Camera AGREEMENT
# Purpose: For labelled clips, how often did VLM agree with AI Camera?
# Thesis point: Validates VLM accuracy against ground truth AI Camera labels.
# =============================================================================

print("=" * 70)
print("ANALYTICS 4 — VLM vs AI CAMERA AGREEMENT")
print("=" * 70)

labelled = fused_event_df[fused_event_df["labelled_checked"] == "y"].copy()

print(f"Clips where VLM checked against AI camera label : {len(labelled)}")
print(f"  Agreed (y)                                    : {(labelled['labelled_agreed'] == 'y').sum()}")
print(f"  Disagreed (n)                                 : {(labelled['labelled_agreed'] == 'n').sum()}")

if len(labelled) > 0:
    agree_rate = (labelled["labelled_agreed"] == "y").mean()
    print(f"  Agreement rate                                : {agree_rate*100:.1f}%")

    # Agreed and also found additional events
    agreed_with_extra = labelled[
        (labelled["labelled_agreed"] == "y") &
        (labelled["n_extra_rankable_vlm_events"] > 0)
    ]
    print(f"\n  Of agreed clips, VLM also found additional events: {len(agreed_with_extra)}")
    print(f"  These are events the AI camera missed:")
    display(
        agreed_with_extra.groupby("original_event_description")
        .agg(
            clips          =("clip", "count"),
            extra_events   =("extra_rankable_vlm_events", lambda x: sum([len(i) for i in x])),
        )
        .sort_values("clips", ascending=False)
        .reset_index()
    )

    # Agreement rate by event type
    agree_by_event = (
        labelled.groupby("original_event_description")
        .agg(
            total  =("labelled_agreed", "count"),
            agreed =("labelled_agreed", lambda x: (x=="y").sum()),
            with_extra=("n_extra_rankable_vlm_events", lambda x: (x > 0).sum()),
        )
        .assign(agreement_pct=lambda x: (100 * x["agreed"] / x["total"]).round(1))
        .sort_values("agreement_pct", ascending=False)
        .reset_index()
    )
    print("\nAgreement rate by event type (with additional detections count):")
    display(agree_by_event)

# ANALYTICS 5: CHANGE TYPE BREAKDOWN

In [ ]:
# =============================================================================
# ANALYTICS 5 — VLM CHANGE TYPE BREAKDOWN
# Purpose: What did the VLM do — confirm, contradict, or add new context?
# Thesis point: Shows how VLM enriches telematics beyond simple confirmation.
# =============================================================================
print("=" * 70)
print("ANALYTICS 5 — VLM CHANGE TYPE BREAKDOWN")
print("=" * 70)

change_counts = (
    fused_event_df["vlm_change_type"]
    .value_counts(dropna=False)
    .rename_axis("change_type")
    .reset_index(name="count")
)
change_counts["pct"] = (100 * change_counts["count"] / len(fused_event_df)).round(1)
display(change_counts)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(change_counts["change_type"], change_counts["count"], color=C_BASE, edgecolor="white")
ax.bar_label(bars, labels=[f"{c}  ({p}%)" for c, p in zip(change_counts["count"], change_counts["pct"])], padding=4, fontsize=9)
ax.set_title("VLM Change Type Distribution\n(all clips)", fontsize=11)
ax.set_xlabel("Number of Clips")
plt.tight_layout()
plt.show()

# ── Confirmed breakdown — what events were confirmed ──────────────────────────
print(f"\n{'─'*70}")
print("CONFIRMED — VLM agreed with telematics label")
print(f"{'─'*70}")
confirmed_df = fused_event_df[fused_event_df["vlm_change_type"] == "confirmed"]
print(f"Total confirmed clips: {len(confirmed_df)}")
display(
    confirmed_df.groupby("original_event_description")
    .agg(
        clips              =("clip",              "count"),
        with_extra_events  =("n_extra_rankable_vlm_events", lambda x: (x > 0).sum()),
    )
    .assign(extra_pct=lambda x: (100 * x["with_extra_events"] / x["clips"]).round(1))
    .sort_values("clips", ascending=False)
    .reset_index()
    .rename(columns={
        "original_event_description": "Original Event",
        "clips"                     : "Clips",
        "with_extra_events"         : "Also found extra",
        "extra_pct"                 : "Extra %",
    })
)

# ── Not confirmed breakdown — what events were removed ────────────────────────
print(f"\n{'─'*70}")
print("NOT CONFIRMED — VLM disagreed, original telematics event removed")
print(f"{'─'*70}")
not_confirmed_df = fused_event_df[fused_event_df["vlm_change_type"] == "not_confirmed"]
print(f"Total not confirmed clips: {len(not_confirmed_df)}")
display(
    not_confirmed_df.groupby("original_event_description")
    .agg(
        clips_removed      =("clip", "count"),
        unique_vehicles    =("vehicle_id", "nunique"),
    )
    .sort_values("clips_removed", ascending=False)
    .reset_index()
    .rename(columns={
        "original_event_description": "Original Event",
        "clips_removed"             : "Clips Removed",
        "unique_vehicles"           : "Unique Vehicles",
    })
)

# ── Contradicted breakdown ────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print("CONTRADICTED — VLM detected something different, original replaced")
print(f"{'─'*70}")
contradicted_df = fused_event_df[fused_event_df["vlm_change_type"] == "contradicted"]
print(f"Total contradicted clips: {len(contradicted_df)}")
display(
    contradicted_df.groupby("original_event_description")
    .agg(
        clips           =("clip",                "count"),
        unique_vehicles =("vehicle_id",          "nunique"),
        vlm_detected    =("primary_vlm_event",   lambda x: x.value_counts().to_dict()),
    )
    .sort_values("clips", ascending=False)
    .reset_index()
    .rename(columns={
        "original_event_description": "Original Event",
        "clips"                     : "Clips",
        "unique_vehicles"           : "Unique Vehicles",
        "vlm_detected"              : "VLM Detected Instead",
    })
)

- confirmed (657) — VLM agreed with what telematics flagged. The original event was in the VLM's detections and nothing extra was found.
- refined (463) — VLM agreed with the original event AND detected additional events on top of it. For example, telematics said v_phone and VLM also found SEATBELT_D_OFF in the same clip.
- not_confirmed (377) — VLM ran on a labelled clip, found nothing matching the original telematics event, and produced no positive detection. The original event was removed from raw_df_vlm.
- contradicted (266) — the VLM disagreed with the original telematics label, but the VLM still detected something — just a different event.
- new_context_added (1670) — The clip had no original telematics label (or the label didn't match anything), but the VLM detected something anyway. These are purely VLM-injected events with no prior telematics signal.
- review_only (7) — VLM didn't produce a definitive detection but put something in the rev codes for human review. Not used in rankings.
- no_vlm_signal (774) — VLM ran but produced no detections at all. Nothing changed for these clips.

# ANALYTICS 6: CAMERA COVERAGE ANALYSIS

In [ ]:
# =============================================================================
# ANALYTICS 6 — CAMERA COVERAGE ANALYSIS
# Purpose: Identify vehicles with high camera covered ratios.
# Thesis point: Separates monitoring integrity violations from driving behaviour.
# =============================================================================

print("=" * 70)
print("ANALYTICS 6 — CAMERA COVERAGE ANALYSIS")
print("=" * 70)

cam_df = final_monthly_rankings_df[final_monthly_rankings_df["total_clips"] > 0].copy()

print(f"Vehicles with camera footage        : {len(cam_df)}")
print(f"Vehicles with covered_ratio > 0.5   : {(cam_df['covered_ratio'] > 0.5).sum()}")
print(f"Vehicles with covered_ratio > 0.8   : {(cam_df['covered_ratio'] > 0.8).sum()}")

print("\nTop 10 vehicles by covered_ratio:")
display(
    cam_df[["vehicle_id","covered_ratio","covered_clips","total_clips","vehicle_behaviour_class_vlm","rank_vlm"]]
    .sort_values("covered_ratio", ascending=False).head(10).reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(cam_df["covered_ratio"], bins=30, color=C_BASE, edgecolor="white", alpha=0.85)
ax.axvline(0.5, color="red",     linewidth=1.5, linestyle="--", label="50% threshold")
ax.axvline(0.8, color="darkred", linewidth=1.5, linestyle="--", label="80% threshold")
ax.legend()
ax.set_title("Camera Covered Ratio Distribution\n(per vehicle)", fontsize=11)
ax.set_xlabel("Covered Ratio (covered clips / total clips)")
ax.set_ylabel("Number of Vehicles")
plt.tight_layout()
plt.show()

# ── Tier 2 review clips ───────────────────────────────────────────────────────
print(f"\nTier 2 — possible covering (cream/beads/blur, VLM ran):")
tier2_in_final = fused_event_df[
    fused_event_df["rev_codes"].apply(
        lambda x: 67 in x if isinstance(x, list) else False
    )
][["clip","vehicle_id","cam_s","why","seatbelt","phone","distraction"]].reset_index(drop=True)

print(f"  Total tier 2 clips : {len(tier2_in_final)}")
if len(tier2_in_final) > 0:
    display(tier2_in_final)

print(f"\nNote: covered_ratio in rankings reflects tier 1 only (genuine covering).")
print(f"Tier 2 clips ran through VLM — any detections are included in rankings.")

# ANALYTICS 7: TOP 20 RISK DRIVERS

In [ ]:
# =============================================================================
# ANALYTICS 7 — TOP 20 RISK DRIVERS: BASELINE vs VLM
# Purpose: Side-by-side baseline vs VLM ranking for top 20 high-risk drivers.
# Thesis point: Shows which drivers VLM identifies as higher risk than
#               telematics alone would suggest.
# =============================================================================

print("=" * 70)
print("ANALYTICS 7 — TOP 20 RISK DRIVERS: BASELINE vs VLM")
print("=" * 70)

top20 = (
    final_monthly_rankings_df[
        final_monthly_rankings_df["vehicle_behaviour_class_vlm"] != "Insufficient Exposure"
    ]
    .sort_values("rank_vlm", ascending=True)
    .drop_duplicates(subset=["vehicle_id"], keep="first")
    .head(20)
    [["vehicle_id","rank_base","rank_vlm","rank_delta",
      "ubi_proxy_score_base","ubi_proxy_score_vlm","score_delta",
      "vehicle_behaviour_class_base","vehicle_behaviour_class_vlm","covered_ratio"]]
    .reset_index(drop=True)
)
display(top20)

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(monthly_rank_compare_df["rank_base"], monthly_rank_compare_df["rank_vlm"],
           alpha=0.5, s=20, color=C_BASE, label="No significant change")

# rank_delta = rank_base - rank_vlm
# positive rank_delta = rank_vlm is lower number = improved (fewer violations found)
# negative rank_delta = rank_vlm is higher number = worsened (more violations found)

big_movers_worse  = monthly_rank_compare_df[monthly_rank_compare_df["rank_delta"] <= -5]  # VLM found MORE
big_movers_better = monthly_rank_compare_df[monthly_rank_compare_df["rank_delta"] >= 5]   # VLM found FEWER

ax.scatter(big_movers_worse["rank_base"],  big_movers_worse["rank_vlm"],
           alpha=0.9, s=60, color=C_VLM,
           label=f"VLM found MORE violations — rank worse ({len(big_movers_worse)} vehicles)")
ax.scatter(big_movers_better["rank_base"], big_movers_better["rank_vlm"],
           alpha=0.9, s=60, color=C_DIFF,
           label=f"VLM found FEWER violations — rank better ({len(big_movers_better)} vehicles)")
max_rank = monthly_rank_compare_df[["rank_base","rank_vlm"]].max().max()

ax.text(max_rank*0.55, max_rank*0.88,
        "VLM found MORE violations\n(rank got worse — above diagonal)",
        fontsize=8, color=C_VLM, alpha=0.85)
ax.text(max_rank*0.05, max_rank*0.10,
        "VLM found FEWER violations\n(rank improved — below diagonal)",
        fontsize=8, color="green", alpha=0.85)

ax.plot([1, max_rank], [1, max_rank], "k--", linewidth=1, alpha=0.4)
ax.set_title("Telematics Rank vs VLM-Adjusted Rank\nDots on diagonal = no change  |  Higher rank number = worse driver", fontsize=10)
ax.set_xlabel("Telematics-only Rank  (rank 1 = worst driver)")
ax.set_ylabel("VLM-Adjusted Rank  (rank 1 = worst driver)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ANALYTICS 8: TELEMATICS vs VLM DETECTION OVERLAP

In [ ]:
# =============================================================================
# ANALYTICS 8 — CLIP-LEVEL AGREEMENT: TELEMATICS AI CAMERA vs VLM
# Purpose: For clips that were processed by the VLM, how often did the
#          telematics AI camera and VLM agree or disagree on each behaviour?
#
# Definitions (all counts are clips, not raw events):
#   "Camera only"  — telematics AI camera flagged the behaviour in this clip,
#                    VLM reviewed the same clip and did not confirm it
#   "VLM only"     — VLM detected the behaviour, telematics did not flag it
#   "Both agreed"  — telematics flagged it AND VLM confirmed it
#
# Note: Seatbelt is included to show it has zero clip-level detections —
#       SEATBELT_D_OFF is sensor-based and produces no dashcam clip,
#       so the VLM never processed it. It still scores via telematics.
# =============================================================================
print("=" * 70)
print("ANALYTICS 8 — CLIP-LEVEL AGREEMENT: TELEMATICS AI CAMERA vs VLM")
print("=" * 70)

behaviour_map = {
    "Phone"      : ("v_phone",        "phone_flag"),
    "Seatbelt"   : ("SEATBELT_D_OFF", "seatbelt_off_flag"),
    "Fatigue"    : ("v_fatigue",      "fatigue_flag"),
    "Distraction": ("v_distraction",  "distraction_flag"),
    "Smoking"    : ("v_smoke",        "smoke_flag"),
}

overlap_rows = []
for label, (tele_event, vlm_flag) in behaviour_map.items():
    cam_only = fused_event_df[(fused_event_df["original_event_description"] == tele_event) & (fused_event_df[vlm_flag] == 0)]
    vlm_only = fused_event_df[(fused_event_df["original_event_description"] != tele_event) & (fused_event_df[vlm_flag] == 1)]
    both     = fused_event_df[(fused_event_df["original_event_description"] == tele_event) & (fused_event_df[vlm_flag] == 1)]
    total    = len(cam_only) + len(vlm_only) + len(both)

    note = ""
    if label == "Seatbelt" and total == 0:
        note = "sensor-based, no clip"

    overlap_rows.append({
        "Behaviour"        : label,
        "Camera only"      : len(cam_only),
        "VLM only"         : len(vlm_only),
        "Both agreed"      : len(both),
        "Total clips"      : total,
        "Note"             : note,
    })

overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)

print("\nNote: 'Camera only' = telematics flagged, VLM disagreed on same clip")
print("      'VLM only'    = VLM flagged, no telematics event for that clip")
print("      'Both agreed' = strongest signal, confirmed by both systems")
print("      All three groups contribute to final driver rankings.")
print(f"\n      Seatbelt shows zero across all columns — SEATBELT_D_OFF is")
print(f"      triggered by the seatbelt buckle sensor, not the camera.")
print(f"      No clip is recorded, so the VLM never processes it.")
print(f"      Its 1,763 raw events still contribute to driver scores via telematics.")

fig, ax = plt.subplots(figsize=(13, 6))
x, w = range(len(overlap_df)), 0.25
ax.bar([i-w for i in x], overlap_df["Camera only"], width=w, label="Camera only (unconfirmed by VLM)", color=C_BASE, edgecolor="white")
ax.bar([i   for i in x], overlap_df["VLM only"],    width=w, label="VLM only (missed by camera)",     color=C_VLM,  edgecolor="white")
ax.bar([i+w for i in x], overlap_df["Both agreed"], width=w, label="Both agreed",                     color=C_DIFF, edgecolor="white")
ax.set_xticks(list(x))
ax.set_xticklabels(overlap_df["Behaviour"])
ax.set_title(
    "Clip-Level Agreement: Telematics AI Camera vs VLM\n"
    "(scoped to clips processed by the VLM pipeline — Seatbelt shown as sensor-only baseline)",
    fontsize=11,
)
ax.set_ylabel("Number of Clips")
ax.legend()
plt.tight_layout()
plt.show()
print("\n✓ Analytics 1-8 complete")

You should investigate what both agreed and what one said yes and other said no etc. To see differences in models. Lol, below legit looks into the smoking one.

# ANALYTICS 9: SMOKING CLIP REVIEW

In [ ]:
# =============================================================================
# ANALYTICS 9 — SMOKING CLIP REVIEW
# Uses anonymised frames from debug_frames_qwen_anon_clean/
# =============================================================================

print("=" * 70)
print("ANALYTICS 9 — SMOKING CLIP REVIEW")
print("=" * 70)

smoking_detected = fused_event_df[fused_event_df["smoke_flag"] == 1].copy()
smoking_missed   = fused_event_df[
    (fused_event_df["original_event_description"] == "v_smoke") &
    (fused_event_df["smoke_flag"] == 0)
].copy()

print(f"\nA. VLM detected smoking          : {len(smoking_detected)} clips")
print(f"B. Telematics flagged, VLM missed: {len(smoking_missed)} clips")

def find_clip_dir(clip_no: int) -> str:
    clip_name = f"clip_{int(clip_no):03d}"
    for sub in os.listdir(ANON_CLEAN_DIR):
        candidate = os.path.join(ANON_CLEAN_DIR, sub, clip_name)
        if os.path.isdir(candidate):
            return candidate
    return None

def _show_anon_clip_frames(clip_no: int, title: str, max_frames: int = 12):
    clip_dir = find_clip_dir(clip_no)
    if clip_dir is None:
        print(f"  clip_{int(clip_no):03d}: not found in anon_clean")
        return
    frame_paths = sorted(Path(clip_dir).glob("frame_driver_*.jpg"))[:max_frames]
    if not frame_paths:
        print(f"  clip_{int(clip_no):03d}: no frames found")
        return
    fig, axes = plt.subplots(1, len(frame_paths), figsize=(4 * len(frame_paths), 4))
    if len(frame_paths) == 1: axes = [axes]
    for ax, fp in zip(axes, frame_paths):
        try:
            ax.imshow(mpimg.imread(str(fp)))
            ax.set_title(fp.name, fontsize=7)
        except Exception:
            ax.set_facecolor("#222")
        ax.axis("off")
    fig.suptitle(title, fontsize=9, y=1.01)
    plt.tight_layout(pad=0.3)
    plt.show()

print(f"\n{'─'*70}")
print("A. CLIPS WHERE VLM DETECTED SMOKING")
print(f"{'─'*70}")
for _, row in smoking_detected.iterrows():
    clip_no = int(row["clip"])
    _show_anon_clip_frames(clip_no, f"clip_{clip_no:03d}  |  VLM: smoking=y  |  Original: {row.get('original_event_description','')}  |  {row.get('why','')}")

print(f"\n{'─'*70}")
print("B. TELEMATICS SAID SMOKING — VLM DISAGREED")
print(f"{'─'*70}")
for _, row in smoking_missed.iterrows():
    clip_no = int(row["clip"])
    _show_anon_clip_frames(clip_no, f"clip_{clip_no:03d}  |  Telematics: v_smoke  |  VLM smoking={row.get('smoking','')}  |  {row.get('why','')}")

print("\n✓ Smoking review complete")

## ANALYTICS 10 — SPEARMAN RANK CORRELATION (RANKING STABILITY)

In [ ]:
# =============================================================================
# ANALYTICS 10 — SPEARMAN RANK CORRELATION (RANKING STABILITY)
# Purpose: Single-number summary of how similar the baseline and VLM-adjusted
#          rankings are across the full fleet.
# Thesis point: A high rho indicates the VLM makes targeted, selective
#               adjustments rather than wholesale restructuring of the ranking.
#               A statistically significant result confirms the correlation
#               is not due to chance.
# Depends on: monthly_rank_compare_df with columns rank_base, rank_vlm
# =============================================================================

from scipy.stats import spearmanr
import pandas as pd
import numpy as np

print("=" * 70)
print("ANALYTICS 10 — SPEARMAN RANK CORRELATION")
print("=" * 70)

# Use only vehicles that have a valid rank in both systems
# (excludes Insufficient Exposure vehicles — they have no rank position)
rank_df = monthly_rank_compare_df.dropna(subset=["rank_base", "rank_vlm"]).copy()

rho, pvalue = spearmanr(rank_df["rank_base"], rank_df["rank_vlm"])

print(f"\nVehicles included in correlation  : {len(rank_df)}")
print(f"Spearman rho                      : {rho:.4f}")
print(f"p-value                           : {pvalue:.4e}")
print(f"Statistically significant (p<0.05): {'Yes' if pvalue < 0.05 else 'No'}")

# Interpret
if rho >= 0.95:
    interpretation = "Very high — VLM makes targeted adjustments; overall ranking structure preserved."
elif rho >= 0.85:
    interpretation = "High — substantial agreement; meaningful but selective changes."
elif rho >= 0.70:
    interpretation = "Moderate — VLM produces notable reordering of the fleet."
else:
    interpretation = "Low — VLM substantially restructures the ranking."

print(f"\nInterpretation: {interpretation}")

print(f"""
Note on interpretation:
  Rank 1 = highest-risk driver (worst).
  Rank {int(rank_df['rank_base'].max())} = lowest-risk driver (best).
  rho close to 1.0 means VLM preserves the telematics ordering;
  rho meaningfully below 1.0 reflects genuine reordering by VLM detections.
""")




## ANALYTICS 11 — RANK MOVEMENT SENSITIVITY TABLE

In [ ]:
# =============================================================================
# ANALYTICS 11 — RANK MOVEMENT SENSITIVITY TABLE
# Purpose: How many drivers moved by >3, >5, >10 rank positions?
# Thesis point: Provides granular evidence of VLM impact beyond averages —
#               shows exactly how many drivers experience material rank changes
#               that would affect managerial decisions.
# =============================================================================

print("=" * 70)
print("ANALYTICS 11 — RANK MOVEMENT SENSITIVITY TABLE")
print("=" * 70)

# rank_delta = rank_base - rank_vlm
# positive rank_delta → VLM rank number is lower → driver improved (fewer violations)
# negative rank_delta → VLM rank number is higher → driver worsened (more violations found)

abs_delta = rank_df["rank_delta"].abs()

thresholds = [1, 3, 5, 10]
rows = []
for t in thresholds:
    n_moved        = (abs_delta >= t).sum()
    n_worsened     = (rank_df["rank_delta"] <= -t).sum()   # VLM found more violations
    n_improved     = (rank_df["rank_delta"] >= t).sum()    # VLM found fewer violations
    pct_of_fleet   = 100 * n_moved / len(rank_df)
    rows.append({
        "Movement threshold"    : f"≥{t} positions",
        "Total moved"           : int(n_moved),
        "% of fleet"            : f"{pct_of_fleet:.1f}%",
        "VLM worsened (↑ risk)" : int(n_worsened),
        "VLM improved (↓ risk)" : int(n_improved),
    })

sensitivity_df = pd.DataFrame(rows)
display(sensitivity_df)

# Also show the raw distribution of absolute movements
print("\nAbsolute rank movement distribution:")
bins = [0, 1, 3, 5, 10, float("inf")]
labels = ["0 (unchanged)", "1–2", "3–4", "5–9", "10+"]
rank_df["movement_band"] = pd.cut(abs_delta, bins=bins, labels=labels, right=False)
band_counts = (
    rank_df["movement_band"]
    .value_counts()
    .reindex(labels)
    .reset_index()
)
band_counts.columns = ["Movement band (positions)", "Vehicles"]
band_counts["% of fleet"] = (100 * band_counts["Vehicles"] / len(rank_df)).round(1).astype(str) + "%"
display(band_counts)

# Visual: bar chart of the sensitivity table
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
x = [r["Movement threshold"] for r in rows]
total  = [r["Total moved"]           for r in rows]
worse  = [r["VLM worsened (↑ risk)"] for r in rows]
better = [r["VLM improved (↓ risk)"] for r in rows]
x_pos = range(len(x))
width = 0.28
ax.bar([p - width for p in x_pos], total,  width, label="Total moved",            color=C_BASE,  edgecolor="white")
ax.bar([p         for p in x_pos], worse,  width, label="VLM worsened (↑ risk)",  color=C_VLM,   edgecolor="white")
ax.bar([p + width for p in x_pos], better, width, label="VLM improved (↓ risk)",  color=C_DIFF,  edgecolor="white")
ax.set_xticks(list(x_pos))
ax.set_xticklabels(x, fontsize=9)
ax.set_title("Rank Movement by Threshold", fontsize=11)
ax.set_ylabel("Number of Vehicles")
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(band_counts["Movement band (positions)"], band_counts["Vehicles"],
       color=C_BASE, edgecolor="white")
ax.set_title("Rank Movement Distribution", fontsize=11)
ax.set_ylabel("Number of Vehicles")
ax.set_xlabel("Rank positions moved (absolute)")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nSummary for thesis:")
print(f"  Spearman rho = {rho:.4f}  (p = {pvalue:.2e})")
print(f"  Vehicles moving ≥3 rank positions  : {(abs_delta >= 3).sum()} "
      f"({100*(abs_delta >= 3).mean():.1f}%)")
print(f"  Vehicles moving ≥5 rank positions  : {(abs_delta >= 5).sum()} "
      f"({100*(abs_delta >= 5).mean():.1f}%)")
print(f"  Vehicles moving ≥10 rank positions : {(abs_delta >= 10).sum()} "
      f"({100*(abs_delta >= 10).mean():.1f}%)")

# PROCESSING TIME ANALYSIS

In [ ]:
# =============================================================================
# CELL 16 — PROCESSING TIME ANALYSIS
# Purpose: Summarise VLM processing times across all clips.
# =============================================================================

print("=" * 70)
print("PROCESSING TIME ANALYSIS")
print("=" * 70)

time_cols = ["t_total_s", "t_llm_s"]
time_cols = [c for c in time_cols if c in fused_event_df.columns]

display(
    fused_event_df[time_cols]
    .apply(pd.to_numeric, errors="coerce")
    .agg(["mean", "median", "min", "max", "count"])
    .round(2)
)

# Distribution of total processing time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, time_cols):
    vals = pd.to_numeric(fused_event_df[col], errors="coerce").dropna()
    ax.hist(vals, bins=40, color=C_BASE, edgecolor="white", alpha=0.85)
    ax.axvline(vals.mean(),   color="red",    linewidth=1.5, linestyle="--", label=f"Mean: {vals.mean():.1f}s")
    ax.axvline(vals.median(), color="orange", linewidth=1.5, linestyle="--", label=f"Median: {vals.median():.1f}s")
    ax.set_title(f"{col} Distribution", fontsize=11)
    ax.set_xlabel("Seconds")
    ax.set_ylabel("Number of Clips")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# EATING, DRINKING & ALCOHOL REVIEW

In [ ]:
# =============================================================================
# CELL 17 — EATING, DRINKING & ALCOHOL REVIEW
# Purpose: Investigate clips where VLM detected eating, drinking or alcohol.
# Thesis note: These detections are excluded from rankings deliberately.
#              Eating/drinking confirmed as real but not alcohol — documented
#              here for transparency. Excluded because:
#              (1) eating/drinking while driving is not a scored telematics event
#              (2) alcohol cannot be confirmed visually with sufficient confidence
# =============================================================================

print("=" * 70)
print("EATING, DRINKING & ALCOHOL REVIEW")
print("=" * 70)

# ── Summary counts ────────────────────────────────────────────────────────────
print("\n=== eatdrink distribution ===")
print(fused_event_df["eatdrink"].value_counts(dropna=False))

print("\n=== drinking distribution ===")
print(fused_event_df["drinking"].value_counts(dropna=False))

print("\n=== alcohol distribution ===")
print(fused_event_df["alcohol"].value_counts(dropna=False))

print("\n=== review_drinking distribution ===")
print(fused_event_df["review_drinking"].value_counts(dropna=False))

# ── High confidence detections ────────────────────────────────────────────────
high_drinking   = fused_event_df[fused_event_df["review_drinking"] == "high"].copy()
medium_drinking = fused_event_df[fused_event_df["review_drinking"] == "medium"].copy()
eatdrink_yes    = fused_event_df[fused_event_df["eatdrink"] == "y"].copy()
alcohol_yes     = fused_event_df[fused_event_df["alcohol"] == "y"].copy()

print(f"\nHigh review_drinking clips   : {len(high_drinking)}")
print(f"Medium review_drinking clips : {len(medium_drinking)}")
print(f"eatdrink=y clips             : {len(eatdrink_yes)}")
print(f"alcohol=y clips              : {len(alcohol_yes)}")

# ── Show top 3 high review_drinking clips ─────────────────────────────────────
print(f"\n{'─'*70}")
print("HIGH REVIEW_DRINKING CLIPS (top 3)")
print(f"{'─'*70}")

for _, row in high_drinking.head(5).iterrows():
    clip_no = int(row["clip"])
    clip_dir = find_clip_dir(clip_no)

    print(f"\nclip_{clip_no:03d}")
    print(f"  vlm_label        : {row.get('vlm_label', '-')}")
    print(f"  event_description: {row.get('original_event_description', '-')}")
    print(f"  drinking         : {row.get('drinking', '-')}")
    print(f"  alcohol          : {row.get('alcohol', '-')}")
    print(f"  eatdrink         : {row.get('eatdrink', '-')}")
    print(f"  review_drinking  : {row.get('review_drinking', '-')}")
    print(f"  why              : {row.get('why', '-')}")

    if clip_dir:
        frame_paths = sorted(Path(clip_dir).glob("frame_driver_*.jpg"))[:12]
        if frame_paths:
            n_cols = min(len(frame_paths), 6)
            n_rows = math.ceil(len(frame_paths) / n_cols)
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
            axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
            for i, ax in enumerate(axes_flat):
                if i < len(frame_paths):
                    try:
                        ax.imshow(mpimg.imread(str(frame_paths[i])))
                        ax.set_title(frame_paths[i].stem, fontsize=7)
                    except Exception:
                        ax.set_facecolor("#222")
                ax.axis("off")
            fig.suptitle(
                f"clip_{clip_no:03d}  |  drinking={row.get('drinking','-')}  "
                f"alcohol={row.get('alcohol','-')}  eatdrink={row.get('eatdrink','-')}  "
                f"review_drinking={row.get('review_drinking','-')}",
                fontsize=8, y=1.01
            )
            plt.tight_layout(pad=0.3)
            plt.show()

# ── Show top 3 eatdrink=y clips ───────────────────────────────────────────────
print(f"\n{'─'*70}")
print("EATDRINK=Y CLIPS (top 3)")
print(f"{'─'*70}")

for _, row in eatdrink_yes.head(3).iterrows():
    clip_no  = int(row["clip"])
    clip_dir = find_clip_dir(clip_no)

    print(f"\nclip_{clip_no:03d}")
    print(f"  eatdrink  : {row.get('eatdrink', '-')}")
    print(f"  drinking  : {row.get('drinking', '-')}")
    print(f"  alcohol   : {row.get('alcohol',  '-')}")
    print(f"  why       : {row.get('why', '-')}")

    if clip_dir:
        frame_paths = sorted(Path(clip_dir).glob("frame_driver_*.jpg"))[:12]
        if frame_paths:
            n_cols = min(len(frame_paths), 6)
            n_rows = math.ceil(len(frame_paths) / n_cols)
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
            axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
            for i, ax in enumerate(axes_flat):
                if i < len(frame_paths):
                    try:
                        ax.imshow(mpimg.imread(str(frame_paths[i])))
                        ax.set_title(frame_paths[i].stem, fontsize=7)
                    except Exception:
                        ax.set_facecolor("#222")
                ax.axis("off")
            fig.suptitle(
                f"clip_{clip_no:03d}  |  eatdrink={row.get('eatdrink','-')}  "
                f"drinking={row.get('drinking','-')}  alcohol={row.get('alcohol','-')}  "
                f"|  {row.get('why','')}",
                fontsize=8, y=1.01
            )
            plt.tight_layout(pad=0.3)
            plt.show()

print("\n✓ Eating/drinking/alcohol review complete")
print("\nNote: These detections are excluded from driver rankings.")
print("Eating/drinking confirmed as real behaviours but not scored.")
print("Alcohol cannot be confirmed visually with sufficient confidence.")

# SCORING WEIGHTS

In [ ]:
# =============================================================================
# CELL 18 — SCORING WEIGHTS & JUSTIFICATION
# Purpose: Explicit documentation of scoring weights for thesis defence.
# Thesis point: Weights reflect industry-standard risk prioritisation —
#               speeding and harsh manoeuvres are the leading causes of
#               fleet accidents; camera obstruction weighted lowest as it
#               is a monitoring integrity issue rather than a direct
#               safety behaviour.
# =============================================================================

print("=" * 70)
print("SCORING WEIGHTS — UBI PROXY SCORE")
print("=" * 70)

weights = {
    "Harsh manoeuvres (episode-based)"      : 0.20,
    "Long speeding episodes (>1 event)"     : 0.18,
    "Driver distraction (phone, seatbelt)"  : 0.13,
    "Short speeding episodes (single event)": 0.12,
    "Fatigue (eye closed, yawn, fatigue)"   : 0.12,
    "Situational risk (headway, lane)"      : 0.10,
    "Power violations"                      : 0.10,
    "Camera obstruction"                    : 0.05,
}

weights_df = pd.DataFrame(
    weights.items(),
    columns=["Behaviour Category", "Weight"]
).sort_values("Weight", ascending=False).reset_index(drop=True)
weights_df["Weight %"] = (weights_df["Weight"] * 100).round(0).astype(int).astype(str) + "%"
weights_df["Justification"] = [
    "Leading cause of fleet accidents — cornering, braking, acceleration",
    "Sustained speeding indicates deliberate risk-taking behaviour",
    "Phone use and seatbelt non-compliance directly increase injury severity",
    "Isolated speed events — lower severity than sustained speeding",
    "Fatigue-related crashes disproportionately fatal",
    "Forward collision, lane departure — situational risk indicators",
    "External battery disconnect may indicate tampering or avoidance",
    "Monitoring integrity issue — not a direct driving behaviour",
]

display(weights_df)

print(f"\nTotal weight : {sum(weights.values()):.2f}  (must sum to 1.0)")
print(f"\nKey design decisions:")
print(f"  • Episode-based aggregation used throughout — prevents high-frequency")
print(f"    drivers from being unfairly penalised for repeated events in one session")
print(f"  • All scores normalised via _rank01() (percentile rank) — absolute")
print(f"    counts don't matter, only relative position within the fleet")
print(f"  • Night driving and exposure removed from scoring after manager feedback —")
print(f"    used only for Insufficient Exposure eligibility gate")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(weights_df["Behaviour Category"], weights_df["Weight"] * 100,
               color=C_BASE, edgecolor="white")
ax.bar_label(bars, labels=weights_df["Weight %"], padding=4, fontsize=9)
ax.set_title("UBI Proxy Score — Behaviour Weights", fontsize=11)
ax.set_xlabel("Weight (%)")
ax.axvline(12.5, color="red", linewidth=1, linestyle="--", alpha=0.4, label="Equal weight reference (12.5%)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# INSUFFICIENT EXPOSURE ANALYSIS

In [ ]:
# =============================================================================
# CELL 19 — INSUFFICIENT EXPOSURE THRESHOLD ANALYSIS
# Purpose: Show how many vehicles fall below the exposure gate and justify
#          the 3-hour cutoff choice.
# Thesis point: Vehicles with very low driving hours cannot be reliably ranked
#               — their scores reflect too few events to be statistically
#               meaningful. The 3-hour threshold was chosen as the minimum
#               window in which all behaviour categories could plausibly occur.
# =============================================================================

print("=" * 70)
print("INSUFFICIENT EXPOSURE ANALYSIS")
print("=" * 70)

# Pull drive hours from VLM-adjusted pipeline output
features_monthly_vlm = vlm_outputs["features_monthly"].copy()

print(f"Insufficient exposure threshold : {INSUFFICIENT_EXPOSURE_HOURS} hours")
print(f"\nVLM-adjusted rankings:")

insufficient = scored_monthly_vlm[scored_monthly_vlm["insufficient_exposure_flag"] == 1]
sufficient   = scored_monthly_vlm[scored_monthly_vlm["insufficient_exposure_flag"] == 0]

print(f"  Vehicles ranked (sufficient exposure)   : {len(sufficient)}")
print(f"  Vehicles excluded (insufficient exposure): {len(insufficient)}")
print(f"  Exclusion rate                           : {100*len(insufficient)/len(scored_monthly_vlm):.1f}%")

print(f"\nDrive hours summary (sufficient exposure vehicles):")
print(sufficient["drive_hours"].describe().round(2))

print(f"\nDrive hours summary (insufficient exposure vehicles):")
print(insufficient["drive_hours"].describe().round(2))

# Visualise drive hours distribution with threshold line
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
all_hours = scored_monthly_vlm["drive_hours"].dropna()
ax.hist(all_hours, bins=40, color=C_BASE, edgecolor="white", alpha=0.85)
ax.axvline(INSUFFICIENT_EXPOSURE_HOURS, color="red", linewidth=2,
           linestyle="--", label=f"Threshold: {INSUFFICIENT_EXPOSURE_HOURS}h")
ax.axvline(all_hours.mean(), color="orange", linewidth=1.5,
           linestyle="--", label=f"Mean: {all_hours.mean():.1f}h")
ax.set_title("Drive Hours Distribution\n(all vehicles)", fontsize=11)
ax.set_xlabel("Drive Hours")
ax.set_ylabel("Number of Vehicles")
ax.legend(fontsize=8)

# Show what happens to scores at different thresholds
ax = axes[1]
thresholds = [1, 2, 3, 4, 5, 6, 8, 10]
excluded_counts = [
    (scored_monthly_vlm["drive_hours"] < t).sum()
    for t in thresholds
]
ax.plot(thresholds, excluded_counts, marker="o", color=C_VLM, linewidth=2)
ax.axvline(INSUFFICIENT_EXPOSURE_HOURS, color="red", linewidth=2,
           linestyle="--", label=f"Chosen threshold: {INSUFFICIENT_EXPOSURE_HOURS}h")
ax.set_title("Vehicles Excluded at Different Thresholds", fontsize=11)
ax.set_xlabel("Threshold (hours)")
ax.set_ylabel("Vehicles Excluded")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nJustification for {INSUFFICIENT_EXPOSURE_HOURS}-hour threshold:")
print(f"  • Below 3 hours, a vehicle has insufficient opportunity for all")
print(f"    behaviour categories to manifest — scores become noise-dominated")
print(f"  • At 3 hours, {(scored_monthly_vlm['drive_hours'] >= 3).sum()} of")
print(f"    {len(scored_monthly_vlm)} vehicles are retained ({100*(scored_monthly_vlm['drive_hours'] >= 3).mean():.1f}%)")
print(f"  • The threshold was validated with fleet management — 3 hours")
print(f"    represents a typical minimum shift duration in this fleet")

#  UNCERTAINTY FLAG ANALYSIS

In [ ]:
# =============================================================================
# CELL 20 — VLM UNCERTAINTY FLAG ANALYSIS
# Purpose: Show how often VLM was uncertain per behaviour field and whether
#          uncertainty correlates with low driver visibility scores.
# Thesis point: Uncertainty is expected in low-visibility clips — the VLM
#               correctly hedges when it cannot see the driver clearly.
#               This validates the model's calibration.
# =============================================================================

print("=" * 70)
print("VLM UNCERTAINTY FLAG ANALYSIS")
print("=" * 70)

uncertain_cols = {
    "seatbelt_uncertain_flag"   : "Seatbelt",
    "phone_uncertain_flag"      : "Phone",
    "distraction_uncertain_flag": "Distraction",
    "fatigue_uncertain_flag"    : "Fatigue",
    "smoking_uncertain_flag"    : "Smoking",
}

# Only non-gated clips — VLM actually ran on these
non_gated = fused_event_df[~fused_event_df["cam_s"].isin([2, 3, 4])].copy()
non_gated["driver_vis"] = pd.to_numeric(non_gated["driver_vis"], errors="coerce")

print(f"Non-gated clips (VLM ran)  : {len(non_gated)}")
print(f"\nUncertainty rates per field:")

uncertainty_rows = []
for col, label in uncertain_cols.items():
    if col not in non_gated.columns:
        continue
    n_uncertain = non_gated[col].sum()
    pct         = 100 * n_uncertain / len(non_gated)

    # Mean driver_vis for uncertain vs certain clips
    vis_uncertain = non_gated.loc[non_gated[col] == 1, "driver_vis"].mean()
    vis_certain   = non_gated.loc[non_gated[col] == 0, "driver_vis"].mean()

    uncertainty_rows.append({
        "Behaviour"          : label,
        "Uncertain clips"    : int(n_uncertain),
        "Uncertain %"        : round(pct, 1),
        "Mean vis (uncertain)": round(vis_uncertain, 3) if pd.notna(vis_uncertain) else None,
        "Mean vis (certain)" : round(vis_certain,   3) if pd.notna(vis_certain)   else None,
    })
    print(f"  {label:<15}: {n_uncertain:>4} clips ({pct:.1f}%)  "
          f"avg driver_vis uncertain={vis_uncertain:.3f}  certain={vis_certain:.3f}")

uncertainty_df = pd.DataFrame(uncertainty_rows)
display(uncertainty_df)

# ── Correlation: driver_vis vs uncertainty ────────────────────────────────────
print(f"\nCorrelation between driver_vis and uncertainty flags:")
for col, label in uncertain_cols.items():
    if col not in non_gated.columns: continue
    corr = non_gated["driver_vis"].corr(non_gated[col])
    print(f"  {label:<15}: r = {corr:.3f}")

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: uncertainty counts per field
ax = axes[0]
bars = ax.barh(uncertainty_df["Behaviour"], uncertainty_df["Uncertain %"],
               color=C_VLM, edgecolor="white")
ax.bar_label(bars, labels=[f"{p}%" for p in uncertainty_df["Uncertain %"]], padding=4, fontsize=9)
ax.set_title("VLM Uncertainty Rate per Behaviour Field\n(non-gated clips only)", fontsize=11)
ax.set_xlabel("Uncertain %")

# Right: driver_vis distribution for uncertain vs certain clips
ax = axes[1]
vis_certain_vals   = non_gated.loc[non_gated["seatbelt_uncertain_flag"] == 0, "driver_vis"].dropna()
vis_uncertain_vals = non_gated.loc[non_gated["seatbelt_uncertain_flag"] == 1, "driver_vis"].dropna()
ax.hist(vis_certain_vals,   bins=20, alpha=0.7, color=C_BASE, edgecolor="white", label="Certain (seatbelt)")
ax.hist(vis_uncertain_vals, bins=20, alpha=0.7, color=C_VLM,  edgecolor="white", label="Uncertain (seatbelt)")
ax.set_title("Driver Visibility Score\nCertain vs Uncertain (seatbelt example)", fontsize=11)
ax.set_xlabel("driver_vis score")
ax.set_ylabel("Number of Clips")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── Driver vis breakdown ──────────────────────────────────────────────────────
print(f"\ndriver_vis distribution (non-gated clips):")
print(non_gated["driver_vis"].value_counts(dropna=False).sort_index())
print(f"\ndriver_vis = 0 (driver not visible) : {(non_gated['driver_vis'] == 0).sum()} clips")
print(f"driver_vis = 1 (driver visible)      : {(non_gated['driver_vis'] == 1).sum()} clips")

# FLEET CAMERA COVERAGE GAP

In [ ]:
# =============================================================================
# CELL 21 — FLEET CAMERA COVERAGE GAP
# Purpose: How many of the 251 fleet vehicles had at least one clip processed?
#          Show coverage distribution, rank movement correlation with coverage.
# Thesis point: VLM analysis is limited to vehicles with camera footage —
#               vehicles without clips cannot be adjusted. Coverage gap
#               quantifies the ceiling on VLM impact.
# =============================================================================

print("=" * 70)
print("FLEET CAMERA COVERAGE GAP")
print("=" * 70)

total_fleet_vehicles = scored_monthly_vlm["vehicle_id"].nunique()

# Vehicles with at least one clip
vehicles_with_clips = fused_event_df["vehicle_id"].dropna().astype("int64").unique()
vehicles_no_clips   = set(scored_monthly_vlm["vehicle_id"].astype("int64").unique()) - set(vehicles_with_clips)

print(f"Total fleet vehicles         : {total_fleet_vehicles}")
print(f"Vehicles with ≥1 clip        : {len(vehicles_with_clips)} ({100*len(vehicles_with_clips)/total_fleet_vehicles:.1f}%)")
print(f"Vehicles with no clips       : {len(vehicles_no_clips)} ({100*len(vehicles_no_clips)/total_fleet_vehicles:.1f}%)")

# Clips per vehicle
clips_per_vehicle = (
    fused_event_df.groupby("vehicle_id")
    .size()
    .reset_index(name="n_clips")
)
clips_per_vehicle["vehicle_id"] = clips_per_vehicle["vehicle_id"].astype("int64")

print(f"\nClips per vehicle (for vehicles with clips):")
print(clips_per_vehicle["n_clips"].describe().round(1))

# ── Rank movement vs clips coverage ──────────────────────────────────────────
coverage_rank = monthly_rank_compare_df.merge(
    clips_per_vehicle, on="vehicle_id", how="left"
).fillna({"n_clips": 0})

coverage_rank["has_clips"] = (coverage_rank["n_clips"] > 0).astype(int)
coverage_rank["abs_rank_delta"] = coverage_rank["rank_delta"].abs()

print(f"\nRank movement summary:")
print(f"  Vehicles WITH clips:")
with_clips_df = coverage_rank[coverage_rank["has_clips"] == 1]
print(f"    Mean abs rank delta  : {with_clips_df['abs_rank_delta'].mean():.2f}")
print(f"    Max rank delta       : {with_clips_df['rank_delta'].max():.0f}")
print(f"    Min rank delta       : {with_clips_df['rank_delta'].min():.0f}")
print(f"    Non-zero score delta : {(with_clips_df['score_delta'] != 0).sum()}")

print(f"\n  Vehicles WITHOUT clips:")
no_clips_df = coverage_rank[coverage_rank["has_clips"] == 0]
print(f"    Mean abs rank delta  : {no_clips_df['abs_rank_delta'].mean():.2f}")
print(f"    Non-zero score delta : {(no_clips_df['score_delta'] != 0).sum()}")

# Correlation between n_clips and abs rank delta
corr = coverage_rank["n_clips"].corr(coverage_rank["abs_rank_delta"])
print(f"\nCorrelation: n_clips vs abs rank delta : r = {corr:.3f}")

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Left: clips per vehicle distribution
ax = axes[0]
ax.hist(clips_per_vehicle["n_clips"], bins=30, color=C_BASE, edgecolor="white", alpha=0.85)
ax.set_title("Clips per Vehicle Distribution\n(vehicles with ≥1 clip)", fontsize=11)
ax.set_xlabel("Number of Clips")
ax.set_ylabel("Number of Vehicles")

# Middle: coverage pie
ax = axes[1]
ax.pie(
    [len(vehicles_with_clips), len(vehicles_no_clips)],
    labels=[f"With clips\n({len(vehicles_with_clips)})", f"No clips\n({len(vehicles_no_clips)})"],
    colors=[C_BASE, "#BDBDBD"],
    autopct="%1.1f%%", startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
ax.set_title("Fleet Camera Coverage\n(% of vehicles with ≥1 clip processed)", fontsize=11)

# Right: scatter n_clips vs abs rank delta
ax = axes[2]
ax.scatter(
    coverage_rank[coverage_rank["n_clips"] > 0]["n_clips"],
    coverage_rank[coverage_rank["n_clips"] > 0]["abs_rank_delta"],
    alpha=0.5, s=20, color=C_VLM
)
ax.set_title(f"Clips Processed vs Rank Movement\n(r = {corr:.3f})", fontsize=11)
ax.set_xlabel("Number of Clips Processed")
ax.set_ylabel("Absolute Rank Delta")

plt.tight_layout()
plt.show()

# ── Top vehicles by clips processed ──────────────────────────────────────────
print(f"\nTop 10 vehicles by clips processed:")
display(
    clips_per_vehicle
    .merge(
        monthly_rank_compare_df[["vehicle_id","rank_base","rank_vlm","rank_delta","score_delta",
                                  "vehicle_behaviour_class_base","vehicle_behaviour_class_vlm"]],
        on="vehicle_id", how="left"
    )
    .sort_values("n_clips", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# CAMERA QUALITY RECLASSIFICATION — METHODOLOGICAL NOTE

In [ ]:
# =============================================================================
# CELL 22 — CAMERA QUALITY RECLASSIFICATION — METHODOLOGICAL NOTE
# =============================================================================
# Documents the camera quality mislabelling issue discovered during analysis
# and the corrective action taken. Important for thesis methodology chapter.
# =============================================================================

print("=" * 70)
print("CAMERA QUALITY RECLASSIFICATION — METHODOLOGICAL NOTE")
print("=" * 70)

print("""
ISSUE DISCOVERED:
The original VLM pipeline used a smear detector (detect_smeared_occlusion)
to classify camera quality. The detector used thresholds:
  SMEAR_BLOB_RATIO_T = 0.55  (too low)
  SMEAR_GRAD_P75_T   = 14.0  (too high)

This caused 675 clips to be classified as cam_s=2 (camera covered) and
skipped the VLM entirely. Manual inspection revealed the vast majority
were normal driving scenes with poor lighting, slight blur, or night driving.

EVIDENCE:
  obstruction_max_coverage distribution for all 675 clips:
  - Mean: 0.009  Max: 0.035  (true physical covering would score 0.3-0.9+)
  - No natural threshold separating covered from non-covered clips
  
  Per-frame metrics revealed only 2 distinct clusters:
  - Genuine covering: blob_ratio > 0.90, grad_p75 < 5
  - Everything else: blob_ratio 0.55-0.82, grad_p75 6-14

CORRECTION APPLIED:
  Three-tier system introduced:
  - Tier 1: blob_ratio >= 0.90 AND grad_p75 <= 5.0
             → cam_s=2, det=[6], VLM skipped (37 clips)
  - Tier 2: mean_lap_roi <= 100 (low texture, possible obstruction)
             → rev=[67], VLM runs with review flag (1 clip)
  - Tier 0: all other clips
             → VLM runs normally (~638 previously missed clips)

IMPACT ON RESULTS:
  Camera covered events in rankings: 675 → 37 (genuine only)
  New clips entering VLM analysis  : ~638
  These 638 clips contributed new detections to the final rankings.
""")

# Show before/after comparison
before_after = pd.DataFrame({
    "Metric"                          : [
        "Clips classified as camera covered",
        "Clips with VLM analysis skipped",
        "Clips entering VLM for first time (after fix)",
        "Genuine tier 1 covered clips",
        "Tier 2 review clips",
    ],
    "Before correction" : [675, 675, 0,   675, 0],
    "After correction"  : [37,  37,  638, 37,  1],
})
display(before_after)

print(f"\nThesis note: All results in this notebook reflect the corrected pipeline.")
print(f"The original mislabelling would have artificially inflated camera")
print(f"obstruction scores and reduced VLM coverage by 638 clips (~15% of fleet).")

# TIER 2 CLIP REVIEW — POSSIBLE CAMERA COVERING

In [ ]:
def show_clip_frames(clip_no: int, title: str = "", max_frames: int = 12):
    """Display anonymised frames for a clip from the clean folder."""
    clip_name = f"clip_{clip_no:03d}"

    # Search across all label subfolders
    clip_dir = None
    for label_dir in Path(ANON_CLEAN_DIR).iterdir():
        candidate = label_dir / clip_name
        if candidate.exists():
            clip_dir = candidate
            break

    if clip_dir is None:
        print(f"  {clip_name}: not found in {ANON_CLEAN_DIR}")
        return

    frames = sorted(clip_dir.glob("frame_driver_*.jpg"))[:max_frames]
    if not frames:
        print(f"  {clip_name}: no frames found in {clip_dir}")
        return

    n_cols = min(6, len(frames))
    n_rows = math.ceil(len(frames) / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.2))
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for i, ax in enumerate(axes_flat):
        if i < len(frames):
            ax.imshow(mpimg.imread(str(frames[i])))
            ax.set_title(frames[i].stem, fontsize=5)
        ax.axis("off")

    fig.suptitle(f"{clip_name}  |  {clip_dir.parent.name}\n{title}", fontsize=8, y=1.01)
    plt.tight_layout(pad=0.3)
    plt.show()

In [ ]:
# Thresholds from Notebook 1 Cell 1 — hardcoded here for reference
SMEAR_BLOB_RATIO_T     = 0.90
SMEAR_GRAD_P75_T       = 5.0
SMEAR_DARK_RATIO_MAX   = 0.85
SMEAR_BRIGHT_RATIO_MAX = 0.85
LAP_ROI_COVERED_T      = 100.0
COVERED_ENTROPY_T      = 3.2
COVERED_EDGE_DENSITY_T = 0.025
COVERED_ROI_STD_T      = 12.0
BLUR_ROI_LAP_VAR_T     = 20.0


# =============================================================================
# CELL 23 — TIER 2 CLIP REVIEW — POSSIBLE CAMERA COVERING
# =============================================================================
print("=" * 70)
print("TIER 2 CLIP REVIEW — POSSIBLE CAMERA COVERING")
print("=" * 70)

tier2_clips_df = fused_event_df[
    fused_event_df["rev_codes"].apply(
        lambda x: 67 in x if isinstance(x, list) else False
    )
].copy()

print(f"Total tier 2 clips: {len(tier2_clips_df)}")

if len(tier2_clips_df) == 0:
    print("No tier 2 clips found.")
else:
    display(tier2_clips_df[[
        "clip","vehicle_id","cam_s","driver_vis",
        "seatbelt","phone","distraction","fatigue","smoking",
        "vlm_change_type","why"
    ]])

    for _, row in tier2_clips_df.iterrows():
        clip_no = int(row["clip"])

        print(f"\n{'═'*60}")
        print(f"clip_{clip_no:03d}  |  vehicle={row.get('vehicle_id','')}  |  why={row.get('why','')}")
        print(f"{'═'*60}")

        # ── Find and load result.json ─────────────────────────────────
        result_path = None
        for label_dir in Path(ANON_CLEAN_DIR).iterdir():
            candidate = label_dir / f"clip_{clip_no:03d}" / "result.json"
            if candidate.exists():
                result_path = candidate
                break

        if not result_path:
            print("  result.json not found — cannot explain classification")
            continue

        with open(result_path) as f:
            result = json.load(f)

        # ── Top-level fields ──────────────────────────────────────────
        cam   = result.get("cam",   {})
        scene = result.get("scene", {})
        meta  = result.get("meta",  {})
        det   = result.get("det",   [])
        rev   = result.get("rev",   [])
        debug = result.get("debug", {})

        cam_o = cam.get("o")
        cam_s = cam.get("s")
        vs    = scene.get("vs")

        print(f"\n  ── Core output ──────────────────────────────────────")
        print(f"  cam.o        = {cam_o}  (1=front, 2=cabin)")
        print(f"  cam.s        = {cam_s}  (0=OK, 2=tier1 covered, 3=blur, 4=foreground, 5=tier2)")
        print(f"  scene.vs     = {vs}  (0=unknown, 1=moving, 2=stopped)")
        print(f"  det          = {det}")
        print(f"  rev          = {rev}")
        print(f"  meta         = {json.dumps(meta, indent=2)}")

        # ── Camera quality debug ──────────────────────────────────────
        cam_dbg = debug.get("cam_quality", {})
        print(f"\n  ── Camera quality debug (decides tier 1 vs tier 2) ──")
        print(f"  tier              = {cam_dbg.get('tier')}")
        print(f"  blob_ratio        = {cam_dbg.get('blob_ratio')}   (tier1 threshold: >{SMEAR_BLOB_RATIO_T})")
        print(f"  grad_p75          = {cam_dbg.get('grad_p75')}     (tier1 threshold: <{SMEAR_GRAD_P75_T})")
        print(f"  dark_ratio        = {cam_dbg.get('dark_ratio')}   (tier1 threshold: <{SMEAR_DARK_RATIO_MAX})")
        print(f"  bright_ratio      = {cam_dbg.get('bright_ratio')} (tier1 threshold: <{SMEAR_BRIGHT_RATIO_MAX})")
        print(f"  lap_roi           = {cam_dbg.get('lap_roi')}      (tier2 threshold: <{LAP_ROI_COVERED_T})")
        print(f"  entropy           = {cam_dbg.get('entropy')}      (tier2 threshold: <{COVERED_ENTROPY_T})")
        print(f"  edge_density      = {cam_dbg.get('edge_density')} (tier2 threshold: <{COVERED_EDGE_DENSITY_T})")
        print(f"  roi_std           = {cam_dbg.get('roi_std')}      (tier2 threshold: <{COVERED_ROI_STD_T})")
        print(f"  blur_roi_lap_var  = {cam_dbg.get('blur_roi_lap_var')} (tier2 threshold: <{BLUR_ROI_LAP_VAR_T})")

        # ── Tier classification explanation ───────────────────────────
        print(f"\n  ── Why tier 2 and not tier 1 or clear? ──────────────")

        blob_ratio = cam_dbg.get("blob_ratio")
        grad_p75   = cam_dbg.get("grad_p75")
        lap_roi    = cam_dbg.get("lap_roi")

        # Tier 1 check
        tier1_blob = blob_ratio is not None and blob_ratio > SMEAR_BLOB_RATIO_T
        tier1_grad = grad_p75   is not None and grad_p75   < SMEAR_GRAD_P75_T

        if tier1_blob and tier1_grad:
            print(f"  → Would be Tier 1 — but cam.s was recorded as 5, check pipeline logic")
        else:
            if blob_ratio is not None:
                print(f"  → blob_ratio={blob_ratio:.3f} — needed >{SMEAR_BLOB_RATIO_T} for tier 1  "
                      f"{'✓ met' if tier1_blob else '✗ not met — ruled out tier 1'}")
            if grad_p75 is not None:
                print(f"  → grad_p75={grad_p75:.3f}   — needed <{SMEAR_GRAD_P75_T} for tier 1  "
                      f"{'✓ met' if tier1_grad else '✗ not met — ruled out tier 1'}")

        # Tier 2 check
        if lap_roi is not None:
            tier2_lap = lap_roi < LAP_ROI_COVERED_T
            print(f"  → lap_roi={lap_roi:.3f}      — needed <{LAP_ROI_COVERED_T} for tier 2  "
                  f"{'✓ met — low texture, possible obstruction' if tier2_lap else '✗ not met'}")

        print(f"\n  Summary: clip did not meet the hard blob+gradient threshold for")
        print(f"  Tier 1 (definite cover), but had low enough texture/laplacian")
        print(f"  variance to be flagged as possibly covered (Tier 2).")
        print(f"  VLM ran and added rev=[67]. It is included in rankings")
        print(f"  with a review flag — not scored as a confirmed cover event.")

        # ── VLM output ────────────────────────────────────────────────
        vlm_dbg = debug.get("vlm", {})
        print(f"\n  ── VLM output ───────────────────────────────────────")
        print(f"  driver_vis   = {vlm_dbg.get('driver_vis')}")
        print(f"  seatbelt     = {vlm_dbg.get('seatbelt')}")
        print(f"  phone        = {vlm_dbg.get('phone')}")
        print(f"  distraction  = {vlm_dbg.get('distraction')}")
        print(f"  fatigue      = {vlm_dbg.get('fatigue')}")
        print(f"  smoking      = {vlm_dbg.get('smoking')}")
        print(f"  why          = {vlm_dbg.get('why')}")
        print(f"  reason       = {vlm_dbg.get('reason')}")

        # ── Motion debug ──────────────────────────────────────────────
        print(f"\n  ── Motion / scene ───────────────────────────────────")
        print(f"  motion_hint  = {debug.get('motion')}")
        motion_detail = debug.get("motion_detail", {})
        print(f"  stats        = {motion_detail.get('stats')}")
        print(f"  first_last   = {motion_detail.get('first_last')}")

        # ── Relevant raw fields only ──────────────────────────────────
        print(f"\n  ── Relevant raw fields ──────────────────────────────")
        print(json.dumps({
            "cam":   result.get("cam"),
            "scene": result.get("scene"),
            "det":   result.get("det"),
            "rev":   result.get("rev"),
            "debug": {
                "motion":      result.get("debug", {}).get("motion"),
                "cam_quality": result.get("debug", {}).get("cam_quality"),
                "vlm": {
                    k: v for k, v in result.get("debug", {}).get("vlm", {}).items()
                    if k in ("driver_vis","seatbelt","phone","distraction",
                             "fatigue","smoking","why","reason")
                },
            },
        }, indent=2))

        show_clip_frames(
            clip_no,
            title=f"clip_{clip_no:03d}  |  Tier 2  |  rev=[67]  |  {row.get('why','')}",
            max_frames=12
        )